# AzSLD 04 — train the word model (Phase 2)

Isolated-sign recognition on landmarks, **signer-independent**: train on the sentence-cut signs
of train signers (200-word vocabulary), report every epoch on the val signers, score the test
signers once at the end with the best-on-val checkpoint. Model: Conv1D + Transformer, ~1.7 M
parameters, exported to ONNX for the browser.

**Inputs:** outputs of 01, the four 02 shards and 03 (*Add Input → Your Work → Notebooks*).
**Kaggle settings:** Accelerator **GPU T4**, Internet **On** (only to install `onnx` if missing).
**Writes** `train/`: `history.csv` (per epoch), `metrics.json` (test), `per_class_test_words.csv`,
`best.pt`, `model.onnx`, `vocab.json`.

Columns per epoch: `W_*` = Words_200 val clips (the benchmark), `S_*` = sentence-cut val signs;
`mF1` = macro-F1. Always compare with the majority-class accuracy printed at the start.

In [ ]:
SMOKE = False          # True: 2 epochs on a sample, to check the setup
CFG = dict(epochs=60, batch=64, lr=1e-3, dim=192, drop=0.2, margin=8, workers=4, seed=0)
# Each entry = one experiment: CFG with these changes. All share the loaded data; results go to
# train/<name>/ and train/summary.csv. Run 3 (split v6, seed 0, all features) is the reference.
# Run 4 (seed1, seed2, hands_only, no_face, epochs100): no_face = hands + pose was best, one seed.
# Run 5: does no_face hold over seeds, and without the face tracker (the "face detected" flag)?
HANDS = ['hands', 'hand_shape', 'hand_vel', 'flags']
NO_FACE = HANDS + ['pose']
RUNS = {
    'no_face_seed1': dict(groups=NO_FACE, seed=1),
    'no_face_seed2': dict(groups=NO_FACE, seed=2),
    'no_face_fd50': dict(groups=NO_FACE, face_drop=0.5),     # flag randomly 0 in training
    'full_fd50': dict(face_drop=0.5),                        # all features, face randomly removed
}

In [ ]:
import glob, json, os, subprocess, sys, tempfile
ON_KAGGLE = os.path.isdir('/kaggle/working')
OUT = '/kaggle/working' if ON_KAGGLE else os.path.abspath('azsl_out')
PKG = os.path.join(tempfile.gettempdir(), 'azsl_pkg')
MODELS = os.path.join(tempfile.gettempdir(), 'azsl_models')
os.makedirs(os.path.join(PKG, 'azsl'), exist_ok=True)
os.makedirs(OUT, exist_ok=True)
if PKG not in sys.path:
    sys.path.insert(0, PKG)

In [ ]:
# ONNX export (for the browser) needs onnx; onnxruntime checks the export against PyTorch.
# Installed only if this image lacks them.
import importlib.util
missing = [p for p in ('onnx', 'onnxruntime') if importlib.util.find_spec(p) is None]
if ON_KAGGLE and missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing])

### The `azsl` package
Generated from `ml/azsl/` — edit it there, not here.

In [ ]:
%%writefile {PKG}/azsl/__init__.py
"""AzSLD → MediaPipe landmarks → signer-independent benchmark.

Shared code for the Kaggle notebooks in ml/notebooks/ (they embed this package
verbatim, see ml/build_notebooks.py). Phases 0–1 of the model plan:

  zenodo     read the AzSLD Zenodo archives by byte range (no 44 GB download)
  landmarks  MediaPipe hands + pose + face, same .task models as the browser
  store      landmark arrays packed into part-*.npz files + manifest.csv
  sources    dataset listings: Sentences annotations, Words_200, Fingerspelling
  split      pseudo-signer clustering and the frozen train/val/test split
  pipeline   the steps the notebooks run
"""

In [ ]:
%%writefile {PKG}/azsl/zenodo.py
"""Read AzSLD from Zenodo without downloading whole archives.

Zenodo answers HTTP Range requests, so the central directory of a zip and any
single member can be fetched on their own. AzSLD_Sentences.zip is 43.9 GB — more
than Kaggle keeps — so recordings are read one at a time and only landmarks stay.
"""
import io
import os
import time
import urllib.error
import urllib.request
import zipfile
import zlib
from dataclasses import dataclass

RECORD = '14222948'
ARCHIVES = {
    'words200': 'AzSLD_Words_200.zip',
    'words100': 'AzSLD_Words_100.zip',
    'fingerspelling': 'AzSLD_Fingerspelling.zip',
    'sentences': 'AzSLD_Sentences.zip',
}
UA = {'User-Agent': 'azsl-landmarks/1.0 (+https://github.com/Chevir-App-Development-Team)'}


def archive_url(key: str) -> str:
    return f'https://zenodo.org/records/{RECORD}/files/{ARCHIVES[key]}?download=1'


def http_get(url: str, start: int | None = None, end: int | None = None,
             retries: int = 8, timeout: int = 120) -> bytes:
    """GET (optionally bytes [start, end)) with backoff — Zenodo rate-limits bursts (429)."""
    headers = dict(UA)
    if start is not None:
        headers['Range'] = f'bytes={start}-{end - 1}'
    last = None
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=timeout) as r:
                data = r.read()
            if start is not None and len(data) != end - start:
                raise IOError(f'short read: {len(data)} of {end - start} bytes')
            return data
        except urllib.error.HTTPError as e:
            if e.code == 404:
                raise
            last, wait = e, min(300, 5 * 2 ** attempt)
            ra = e.headers.get('Retry-After') if e.headers else None
            if ra and ra.isdigit():
                wait = max(wait, int(ra))
        except (urllib.error.URLError, OSError, TimeoutError) as e:
            last, wait = e, min(300, 5 * 2 ** attempt)
        time.sleep(wait)
    raise IOError(f'GET {url} failed after {retries} tries: {last}')


def download(url: str, dest: str, chunk: int = 1 << 24) -> str:
    """Whole-file download in ranged chunks; resumes if `dest` is partly there."""
    total = int(_head_size(url))
    done = os.path.getsize(dest) if os.path.exists(dest) else 0
    with open(dest, 'ab') as f:
        while done < total:
            end = min(done + chunk, total)
            f.write(http_get(url, done, end))
            done = end
    return dest


def _head_size(url: str, retries: int = 8) -> int:
    req = urllib.request.Request(url, headers={**UA, 'Range': 'bytes=0-0'})
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                return int(r.headers['Content-Range'].split('/')[-1])
        except (urllib.error.URLError, OSError, TimeoutError) as e:   # DNS blips, 429, resets
            if attempt == retries - 1:
                raise IOError(f'size of {url}: {e}') from e
            time.sleep(min(300, 5 * 2 ** attempt))


class RangeFile(io.RawIOBase):
    """Seekable read-only view of a remote file; just enough for zipfile to read the directory."""

    def __init__(self, url: str, block: int = 1 << 20):
        self.url, self.block, self.pos, self.cache = url, block, 0, {}
        self.size = _head_size(url)

    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else self.pos + off if whence == 1 else self.size + off
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = max(0, min(n, self.size - self.pos))
        if not n:
            return b''
        if n > 4 * self.block:                     # big reads (the central directory) in one go
            data = http_get(self.url, self.pos, self.pos + n)
        else:
            out, p = bytearray(), self.pos
            while p < self.pos + n:
                b = p // self.block
                if b not in self.cache:
                    s = b * self.block
                    self.cache[b] = http_get(self.url, s, min(s + self.block, self.size))
                chunk, o = self.cache[b], p - b * self.block
                take = min(len(chunk) - o, self.pos + n - p)
                out += chunk[o:o + take]
                p += take
            data = bytes(out)
        self.pos += len(data)
        return data

    def readinto(self, b):
        d = self.read(len(b))
        b[:len(d)] = d
        return len(d)


@dataclass(frozen=True)
class Member:
    """Picklable pointer to one zip member — workers fetch it with a single request."""
    url: str
    name: str
    offset: int          # local header offset
    csize: int
    size: int
    method: int
    crc: int


def list_members(url: str) -> list[Member]:
    z = zipfile.ZipFile(RangeFile(url))
    return [Member(url, i.filename, i.header_offset, i.compress_size, i.file_size, i.compress_type, i.CRC)
            for i in z.infolist() if not i.is_dir()]


def _slack(m: Member) -> int:
    return 30 + len(m.name.encode('utf-8')) + 1024


def _unpack(m: Member, raw: bytes) -> bytes:
    """`raw` starts at the member's local header; returns the verified, decompressed data."""
    if raw[:4] != b'PK\x03\x04':
        raise IOError(f'{m.name}: no local file header at {m.offset}')
    n, e = int.from_bytes(raw[26:28], 'little'), int.from_bytes(raw[28:30], 'little')
    start = 30 + n + e
    data = raw[start:start + m.csize]
    if len(data) < m.csize:
        data = http_get(m.url, m.offset + start, m.offset + start + m.csize)
    if m.method == zipfile.ZIP_DEFLATED:
        data = zlib.decompressobj(-15).decompress(data)
    elif m.method != zipfile.ZIP_STORED:
        raise IOError(f'{m.name}: unsupported compression {m.method}')
    if zlib.crc32(data) != m.crc:
        raise IOError(f'{m.name}: CRC mismatch')
    return data


def read_member(m: Member) -> bytes:
    """Local header + data in one Range request (a second one only if the local extra field is huge)."""
    return _unpack(m, http_get(m.url, m.offset, m.offset + _slack(m) + m.csize))


def read_many(members: list[Member], max_span: int = 8 << 20):
    """Small neighbouring members (annotation JSONs) fetched as shared spans: ~1 request per folder
    instead of one per file. Yields (member, data)."""
    ms = sorted(members, key=lambda m: m.offset)
    i = 0
    while i < len(ms):
        j, end = i, ms[i].offset + _slack(ms[i]) + ms[i].csize
        while j + 1 < len(ms):
            nxt = ms[j + 1]
            nend = nxt.offset + _slack(nxt) + nxt.csize
            if nxt.offset > end + (64 << 10) or nend - ms[i].offset > max_span:
                break
            j, end = j + 1, max(end, nend)
        span = http_get(ms[i].url, ms[i].offset, end)
        for m in ms[i:j + 1]:
            yield m, _unpack(m, span[m.offset - ms[i].offset:])
        i = j + 1

In [ ]:
%%writefile {PKG}/azsl/landmarks.py
"""MediaPipe landmarks per frame, with the same models and settings as the browser.

hand_landmarker.task below is byte-identical to public/tercume/assets/hand_landmarker.task
(same md5), and the hand options match src/tercume/hands.js. Videos run in VIDEO mode
(tracking), like the live camera, so training data has the same jitter and misses
the model will see in the browser. Pose and face use the float16/1 releases that the
browser will load when they are added there.

Per frame:
  hand        (2, 21, 3)  image coords x, y in [0, 1], z relative — slot 0 = signer's RIGHT hand
  hand_world  (2, 21, 3)  metric 3D hand (MediaPipe world landmarks)
  hand_score  (2,)        handedness score of the detection in that slot
  pose        (33, 4)     image coords + visibility
  pose_world  (33, 3)
  face        (102, 3)    lips, eyes, brows, nose, oval, irises (FACE_IDX)
  blend       (52,)       face blendshapes (brows, eyes, mouth — non-manual signals)
Missing detections are NaN. Slots are anatomical: pose wrists decide which hand is which
(same rule as tools/build_words.py), so crossing hands keep their slot.
"""
import base64
import hashlib
import os
import urllib.request

import cv2
import numpy as np

_MP = 'https://storage.googleapis.com/mediapipe-models'
MODELS = {   # name: (url, base64 md5 as GCS reports it)
    'hand': (f'{_MP}/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task', 'FTGEMOo4UWcP6ZFBFqnPrQ=='),
    'pose': (f'{_MP}/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task', 'BKdd33yBGsehpFIyZt19iA=='),
    'face': (f'{_MP}/face_landmarker/face_landmarker/float16/1/face_landmarker.task', 'sOcnSQehZEQE/vZrKN1thQ=='),
}
# OpenCV SFace — only to group recordings by signer for the split; embeddings never leave the split step.
SFACE_URLS = [
    'https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx',
    'https://huggingface.co/opencv/face_recognition_sface/resolve/main/face_recognition_sface_2021dec.onnx',
]
SFACE_FILE = 'face_recognition_sface_2021dec.onnx'
SFACE_MD5 = 'lDaT6px939mmbce36NUqTw=='

FACE_IDX = [
    # lips (outer + inner)
    61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    # eyes
    263, 249, 390, 373, 374, 380, 381, 382, 362, 466, 388, 387, 386, 385, 384, 398,
    33, 7, 163, 144, 145, 153, 154, 155, 133, 246, 161, 160, 159, 158, 157, 173,
    # eyebrows
    276, 283, 282, 295, 285, 300, 293, 334, 296, 336,
    46, 53, 52, 65, 55, 70, 63, 105, 66, 107,
    # nose, face oval extremes (head pose), iris centres
    1, 2, 98, 327, 10, 152, 234, 454, 468, 473,
]
FIELDS = {
    'hand': (2, 21, 3), 'hand_world': (2, 21, 3), 'hand_score': (2,),
    'pose': (33, 4), 'pose_world': (33, 3),
    'face': (len(FACE_IDX), 3), 'blend': (52,),
}
P_NOSE, P_L_SH, P_R_SH, P_L_WR, P_R_WR = 0, 11, 12, 15, 16
EMB_DIM = 128


def _md5_b64(path: str) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return base64.b64encode(h.digest()).decode()


def ensure_models(model_dir: str, sface: bool = False) -> dict:
    """Download the .task files (and SFace) once, verify md5. Returns name → local path."""
    os.makedirs(model_dir, exist_ok=True)
    paths = {}
    for name, (url, md5) in MODELS.items():
        p = os.path.join(model_dir, os.path.basename(url))
        if not os.path.exists(p) or _md5_b64(p) != md5:
            urllib.request.urlretrieve(url, p)
        got = _md5_b64(p)
        if got != md5:
            raise IOError(f'{name}: md5 {got} != {md5} — model release changed, landmarks would not match the browser')
        paths[name] = p
    if sface:
        p = os.path.join(model_dir, SFACE_FILE)
        for url in SFACE_URLS:
            if os.path.exists(p) and _md5_b64(p) == SFACE_MD5:
                break
            try:
                urllib.request.urlretrieve(url, p)
            except OSError as e:
                print(f'SFace download failed from {url}: {e}')
        if not os.path.exists(p) or _md5_b64(p) != SFACE_MD5:
            raise IOError('SFace model missing or md5 mismatch')
        paths['sface'] = p
    return paths


def empty(n: int) -> dict:
    return {k: np.full((n, *s), np.nan, np.float32) for k, s in FIELDS.items()}


def assign_hands(wrists_xy, pose_wrists, aspect):
    """Detected hands → [right slot, left slot] (indices into the detections or None).

    Pose wrists decide, so crossed hands keep their side. Without a pose the signer's
    right hand is on the image's left (the recordings are not mirrored). Same rule as
    tools/build_words.py `_assign`.
    """
    slots = [None, None]
    if not wrists_xy:
        return slots
    if pose_wrists is None:
        for s, k in enumerate(sorted(range(len(wrists_xy)), key=lambda k: wrists_xy[k][0])[:2]):
            slots[s] = k
        return slots
    (lx, ly), (rx, ry) = pose_wrists

    def d(k, px, py):
        return np.hypot((wrists_xy[k][0] - px) * aspect, wrists_xy[k][1] - py)

    if len(wrists_xy) >= 2:
        return [0, 1] if d(0, rx, ry) + d(1, lx, ly) <= d(0, lx, ly) + d(1, rx, ry) else [1, 0]
    slots[0 if d(0, rx, ry) <= d(0, lx, ly) else 1] = 0
    return slots


class Extractor:
    """One per worker process. VIDEO-mode landmarkers are recreated per video (tracking state
    must not leak between clips); IMAGE-mode ones are created once and reused."""

    def __init__(self, paths: dict, face: bool = True, embed: bool = False):
        import mediapipe as mp
        from mediapipe.tasks.python import BaseOptions
        from mediapipe.tasks.python import vision
        self.mp, self.V, self.Base = mp, vision, BaseOptions
        self.paths, self.face_on = paths, face
        self.recognizer = cv2.FaceRecognizerSF.create(paths['sface'], '') if embed else None
        self._image_lms = None

    def _create(self, video: bool):
        V, B = self.V, self.Base
        mode = V.RunningMode.VIDEO if video else V.RunningMode.IMAGE
        hand = V.HandLandmarker.create_from_options(V.HandLandmarkerOptions(
            base_options=B(model_asset_path=self.paths['hand']), running_mode=mode, num_hands=2,
            min_hand_detection_confidence=0.5, min_hand_presence_confidence=0.5,
            min_tracking_confidence=0.5))
        pose = V.PoseLandmarker.create_from_options(V.PoseLandmarkerOptions(
            base_options=B(model_asset_path=self.paths['pose']), running_mode=mode, num_poses=1))
        face = V.FaceLandmarker.create_from_options(V.FaceLandmarkerOptions(
            base_options=B(model_asset_path=self.paths['face']), running_mode=mode, num_faces=1,
            output_face_blendshapes=True)) if self.face_on else None
        return hand, pose, face

    @staticmethod
    def _close(lms):
        for lm in lms:
            if lm is not None:
                lm.close()

    def _frame(self, out, t, rgb, lms, ts):
        """Fills row t of `out`; returns the 478 face landmarks (or None) for embedding."""
        hand, pose, face = lms
        im = self.mp.Image(image_format=self.mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
        if ts is None:
            hr, pr = hand.detect(im), pose.detect(im)
            fr = face.detect(im) if face else None
        else:
            hr, pr = hand.detect_for_video(im, ts), pose.detect_for_video(im, ts)
            fr = face.detect_for_video(im, ts) if face else None
        H, W = rgb.shape[:2]

        wrists = None
        if pr.pose_landmarks:
            pl = pr.pose_landmarks[0]
            out['pose'][t] = [[p.x, p.y, p.z, p.visibility] for p in pl]
            out['pose_world'][t] = [[p.x, p.y, p.z] for p in pr.pose_world_landmarks[0]]
            wrists = (pl[P_L_WR].x, pl[P_L_WR].y), (pl[P_R_WR].x, pl[P_R_WR].y)

        found = hr.hand_landmarks or []
        slots = assign_hands([(h[0].x, h[0].y) for h in found], wrists, W / H)
        for s, k in enumerate(slots):
            if k is None:
                continue
            out['hand'][t, s] = [[p.x, p.y, p.z] for p in found[k]]
            out['hand_world'][t, s] = [[p.x, p.y, p.z] for p in hr.hand_world_landmarks[k]]
            out['hand_score'][t, s] = hr.handedness[k][0].score

        if fr is not None and fr.face_landmarks:
            fl = fr.face_landmarks[0]
            out['face'][t] = [[fl[i].x, fl[i].y, fl[i].z] for i in FACE_IDX]
            if fr.face_blendshapes:
                bs = [c.score for c in fr.face_blendshapes[0]][:FIELDS['blend'][0]]
                out['blend'][t, :len(bs)] = bs
            return fl
        return None

    def video(self, path: str, stride: int = 1, embed_at=(0.25, 0.5, 0.75)):
        """→ (arrays, meta, face) for a video file; face = {'emb', 'thumb'} or None."""
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        n_hint = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
        rows, picks = [], []
        targets = sorted({int(f * n_hint) for f in embed_at}) if (self.recognizer and n_hint) else []
        lms = self._create(video=True)
        try:
            i = -1
            while True:
                ok, bgr = cap.read()
                if not ok:
                    break
                i += 1
                if i % stride:
                    continue
                row = empty(1)
                fl = self._frame(row, 0, cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), lms, int(round(i * 1000.0 / fps)))
                rows.append(row)
                if fl is not None and targets and i >= targets[0]:
                    picks.append((bgr, fl))
                    targets.pop(0)
        finally:
            cap.release()
            self._close(lms)
        arrays = {k: np.concatenate([r[k] for r in rows]) if rows else np.zeros((0, *s), np.float32)
                  for k, s in FIELDS.items()}
        meta = summarize(arrays, W, H)
        meta.update(fps=round(fps / stride, 3), width=W, height=H, n_frames=len(rows))
        return arrays, meta, self._embed(picks)

    def image(self, data: bytes):
        """Single image (fingerspelling) → (arrays with T=1, meta)."""
        bgr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        if bgr is None:
            raise ValueError('could not decode image')
        if self._image_lms is None:
            self._image_lms = self._create(video=False)
        out = empty(1)
        self._frame(out, 0, cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), self._image_lms, None)
        H, W = bgr.shape[:2]
        meta = summarize(out, W, H)
        meta.update(fps=0.0, width=W, height=H, n_frames=1)
        return out, meta

    def _embed(self, picks):
        """Mean SFace embedding over a few frames + one aligned 112×112 crop for eyeballing clusters."""
        if not self.recognizer or not picks:
            return None
        feats, thumb = [], None
        for bgr, fl in picks:
            H, W = bgr.shape[:2]
            pt = lambda i: (fl[i].x * W, fl[i].y * H)
            xs, ys = [p.x * W for p in fl], [p.y * H for p in fl]
            (e1, e2), (m1, m2) = sorted([pt(468), pt(473)]), sorted([pt(61), pt(291)])
            row = np.array([[min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys),
                             *e1, *e2, *pt(1), *m1, *m2, 1.0]], np.float32)
            crop = self.recognizer.alignCrop(bgr, row)
            f = self.recognizer.feature(crop).ravel().astype(np.float32)
            feats.append(f / (np.linalg.norm(f) + 1e-9))
            if thumb is None:
                thumb = cv2.imencode('.jpg', crop, [cv2.IMWRITE_JPEG_QUALITY, 80])[1].tobytes()
        emb = np.mean(feats, axis=0)
        return {'emb': emb / (np.linalg.norm(emb) + 1e-9), 'thumb': thumb}


def summarize(a: dict, W: int, H: int) -> dict:
    """Detection rates + a frontal-camera check (nose offset from shoulder centre / shoulder width)."""
    T = a['hand'].shape[0]
    if not T:
        return dict(hand_rate=0.0, two_hand_rate=0.0, pose_rate=0.0, face_rate=0.0, frontal=float('nan'))
    hands = ~np.isnan(a['hand'][:, :, 0, 0])
    pose_ok = ~np.isnan(a['pose'][:, 0, 0])
    frontal = float('nan')
    if pose_ok.any():
        p = a['pose'][pose_ok]
        sw = np.abs(p[:, P_L_SH, 0] - p[:, P_R_SH, 0]) * W
        off = np.abs(p[:, P_NOSE, 0] - (p[:, P_L_SH, 0] + p[:, P_R_SH, 0]) / 2) * W
        frontal = float(np.median(off / np.maximum(sw, 1.0)))
    return dict(
        hand_rate=round(float(hands.any(1).mean()), 4),
        two_hand_rate=round(float(hands.all(1).mean()), 4),
        pose_rate=round(float(pose_ok.mean()), 4),
        face_rate=round(float((~np.isnan(a['face'][:, 0, 0])).mean()), 4),
        frontal=round(frontal, 4),
    )

In [ ]:
%%writefile {PKG}/azsl/store.py
"""Landmarks on disk: items packed into part-NNNNN.npz files plus one manifest.csv per subset.

A part holds a few hundred items concatenated along time (float16) with `ids` and
`offsets`, so thousands of clips load as a handful of files. Each manifest row says
which part and offset an item lives in, plus its metadata. Writes are resumable:
items already in the manifest are skipped, and a part is written before its rows are
appended, so an interrupted run never leaves manifest rows without data.
"""
import csv
import functools
import glob
import os

import numpy as np

from .landmarks import EMB_DIM, FIELDS

BASE_COLS = ['id', 'part', 'offset', 'length']


class PartWriter:
    def __init__(self, out_dir: str, items_per_part: int = 400, embeddings: bool = False):
        os.makedirs(out_dir, exist_ok=True)
        self.dir, self.per_part, self.embeddings = out_dir, items_per_part, embeddings
        self.manifest = os.path.join(out_dir, 'manifest.csv')
        nums = [int(os.path.basename(p)[5:10]) for p in glob.glob(os.path.join(out_dir, 'part-*.npz'))]
        self.next_part = max(nums, default=-1) + 1
        self.cols = None
        if os.path.exists(self.manifest):
            with open(self.manifest, newline='', encoding='utf-8') as f:
                self.cols = next(csv.reader(f), None)
        self.buf = []

    def done_ids(self) -> set:
        if not os.path.exists(self.manifest):
            return set()
        with open(self.manifest, newline='', encoding='utf-8') as f:
            return {r['id'] for r in csv.DictReader(f)}

    def add(self, item_id: str, arrays: dict, meta: dict, face: dict | None = None):
        self.buf.append((item_id, arrays, meta, face))
        if len(self.buf) >= self.per_part:
            self.flush()

    def flush(self):
        if not self.buf:
            return
        name = f'part-{self.next_part:05d}.npz'
        lengths = [a['hand'].shape[0] for _, a, _, _ in self.buf]
        offsets = np.concatenate([[0], np.cumsum(lengths)]).astype(np.int64)
        payload = {k: np.concatenate([a[k] for _, a, _, _ in self.buf]).astype(np.float16) for k in FIELDS}
        payload['ids'] = np.array([i for i, _, _, _ in self.buf])
        payload['offsets'] = offsets
        if self.embeddings:
            emb = np.full((len(self.buf), EMB_DIM), np.nan, np.float32)
            thumbs, toff = [], [0]
            for j, (_, _, _, face) in enumerate(self.buf):
                if face is not None:
                    emb[j] = face['emb']
                t = (face or {}).get('thumb') or b''
                thumbs.append(np.frombuffer(t, np.uint8))
                toff.append(toff[-1] + len(t))
            payload['emb'] = emb
            payload['thumb'] = np.concatenate(thumbs) if thumbs else np.zeros(0, np.uint8)
            payload['thumb_off'] = np.array(toff, np.int64)
        path = os.path.join(self.dir, name)
        np.savez_compressed(path + '.tmp.npz', **payload)
        os.replace(path + '.tmp.npz', path)

        rows = [{'id': i, 'part': name, 'offset': int(offsets[j]), 'length': lengths[j], **meta}
                for j, (i, _, meta, _) in enumerate(self.buf)]
        if self.cols is None:
            self.cols = BASE_COLS + [k for k in rows[0] if k not in BASE_COLS]
        extra = set(rows[0]) - set(self.cols)
        if extra:
            raise ValueError(f'new manifest columns {extra} — keep metadata keys stable within a subset')
        new = not os.path.exists(self.manifest)
        with open(self.manifest, 'a', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=self.cols)
            if new:
                w.writeheader()
            w.writerows(rows)
        self.buf = []
        self.next_part += 1

    close = flush


class LandmarkStore:
    """Read side. `dirs` may span several outputs (e.g. one per Sentences shard)."""

    def __init__(self, dirs):
        import pandas as pd
        dirs = [dirs] if isinstance(dirs, str) else list(dirs)
        frames = []
        for d in dirs:
            m = os.path.join(d, 'manifest.csv')
            if os.path.exists(m):
                df = pd.read_csv(m, dtype={'id': str}, keep_default_na=False, na_values=[''])
                df['dir'] = d
                frames.append(df)
        if not frames:
            raise FileNotFoundError(f'no manifest.csv in {dirs}')
        self.manifest = pd.concat(frames, ignore_index=True).drop_duplicates('id', keep='last')
        self.index = self.manifest.set_index('id')

    def __len__(self):
        return len(self.manifest)

    @functools.lru_cache(maxsize=16)
    def _part(self, path):
        with np.load(path) as z:
            return {k: z[k] for k in z.files}

    def get(self, item_id: str, dtype=np.float32) -> dict:
        r = self.index.loc[item_id]
        p = self._part(os.path.join(r['dir'], r['part']))
        s, n = int(r['offset']), int(r['length'])
        return {k: p[k][s:s + n].astype(dtype) for k in FIELDS}

    def embeddings(self):
        """→ (ids, emb [N, 128], thumbs list[bytes]) over every part that stored them."""
        ids, embs, thumbs = [], [], []
        for d, part in self.manifest[['dir', 'part']].drop_duplicates().itertuples(index=False):
            p = self._part(os.path.join(d, part))
            if 'emb' not in p:
                continue
            off = p['thumb_off']
            for j, i in enumerate(p['ids']):
                ids.append(str(i))
                embs.append(p['emb'][j])
                thumbs.append(p['thumb'][off[j]:off[j + 1]].tobytes())
        return ids, (np.stack(embs) if embs else np.zeros((0, EMB_DIM), np.float32)), thumbs

In [ ]:
%%writefile {PKG}/azsl/sources.py
"""What is inside the AzSLD archives, as tables.

Layout (checked against Zenodo 14222948, Oct 2026):
  AzSLD_Sentences/<folder>/{Cam1,Cam2}/<YYYY-MM-DD HH-MM-SS>.mp4   + <folder>/ann/<same>.mp4.json
      Cam2 is the FRONTAL camera, and annotations belong to Cam2 (frame counts match);
      Cam1 is the side view and can be a few frames shorter. 1280×960, 30 fps.
      ann JSON (Supervisely): tags[{name, frameRange [start, end] 1-based inclusive, key}]
  AzSLD_Sentences/sentence_list.csv                                folder → sentence, gloss sequence
  AzSLD_Words_200/<GLOSS>/<key>.mp4    clips cut from Cam2 at a tag's frameRange, no margin;
      <key> is the tag key, which links a clip to its recording (and so to its signer)
  AzSLD_Fingerspelling/<LETTER>/*.jpg  40 images per static letter, ~800 frames per dynamic
      letter. "ID--<n>--__<uuid>.jpg" names carry a contributor id; others are numbered frames.
No signer ids anywhere — see split.py for how recordings are grouped by signer.
"""
import csv
import hashlib
import io
import json
import re
from datetime import datetime

from .zenodo import Member, read_many

DYNAMIC_LETTERS = set('çdgköüyz')          # same split as tools/build_poses.py
RUN_GAP = 20                               # frame-number gap that starts a new clip


def az_lower(s: str) -> str:
    """Azerbaijani lower-case: I → ı, İ → i (str.lower() gets these wrong)."""
    return s.replace('I', 'ı').replace('İ', 'i').lower()


def anon(raw: str) -> str:
    """Contributor ids in file names are Telegram user ids — keep only a short hash."""
    return hashlib.sha1(raw.encode()).hexdigest()[:10]


def parse_stamp(stem: str):
    m = re.match(r'(\d{4}-\d\d-\d\d \d\d-\d\d-\d\d)', stem)
    return datetime.strptime(m.group(1), '%Y-%m-%d %H-%M-%S') if m else None


def sentence_recordings(members: list[Member], camera: str = 'Cam2') -> list[Member]:
    """Video members of one camera, in a stable order."""
    rx = re.compile(rf'^AzSLD_Sentences/([^/]+)/{camera}/([^/]+)\.mp4$')
    return sorted((m for m in members if rx.match(m.name)), key=lambda m: m.name)


def rec_id(name: str) -> str:
    """'AzSLD_Sentences/12/Cam2/2022-06-01 15-17-31.mp4' → '12/2022-06-01 15-17-31'."""
    p = name.split('/')
    stem = p[-1]
    for suf in ('.mp4.json', '.mp4'):
        if stem.endswith(suf):
            stem = stem[:-len(suf)]
    return f'{p[1]}/{stem}'


def sentences_index(members: list[Member]):
    """→ (tags, recordings, sentences) DataFrames from the annotation JSONs + sentence_list.csv."""
    import pandas as pd
    anns = [m for m in members if re.match(r'^AzSLD_Sentences/[^/]+/ann/.+\.json$', m.name)]
    lst = [m for m in members if m.name.endswith('sentence_list.csv')]
    cams = {c: {rec_id(m.name): m.size for m in sentence_recordings(members, c)} for c in ('Cam1', 'Cam2')}

    tags, recs = [], []
    for m, data in read_many(anns):
        rid = rec_id(m.name)
        d = json.loads(data.decode('utf-8'))
        recs.append({
            'rec_id': rid, 'folder': rid.split('/')[0], 'stamp': parse_stamp(rid.split('/')[1]),
            'frames': d.get('framesCount'), 'width': d.get('size', {}).get('width'),
            'height': d.get('size', {}).get('height'), 'n_tags': len(d.get('tags', [])),
            'cam1': rid in cams['Cam1'], 'cam2': rid in cams['Cam2'],
        })
        for t in d.get('tags', []):
            fr = t.get('frameRange') or [None, None]
            tags.append({'key': t['key'], 'gloss': t['name'], 'rec_id': rid, 'start': fr[0], 'end': fr[1],
                         'labeler': t.get('labelerLogin'), 'created': t.get('createdAt')})
    tags, recs = pd.DataFrame(tags), pd.DataFrame(recs)
    tags['dup'] = _duplicates(tags)

    sentences = pd.DataFrame(columns=['folder', 'text', 'glosses'])
    if lst:
        rows = list(csv.reader(io.StringIO(next(read_many(lst))[1].decode('utf-8-sig'))))
        sentences = pd.DataFrame([{'folder': r[0].strip(), 'text': r[1].strip(), 'glosses': r[2].strip()}
                                  for r in rows[1:] if len(r) >= 3])
    return tags, recs, sentences


def _duplicates(tags):
    """Same gloss annotated twice over the same span of one recording (seen in the data) → later one is dup."""
    dup = [False] * len(tags)
    if tags.empty:
        return dup
    for _, g in tags.sort_values('created').groupby(['rec_id', 'gloss']):
        kept = []
        for idx, r in g.iterrows():
            s, e = r['start'], r['end']
            if any(min(e, ke) - max(s, ks) + 1 >= 0.5 * min(e - s + 1, ke - ks + 1) for ks, ke in kept):
                dup[tags.index.get_loc(idx)] = True
            else:
                kept.append((s, e))
    return dup


def words_listing(names: list[str], root: str = 'AzSLD_Words_200'):
    """Zip member names → key (tag key, links to a recording), gloss, member."""
    import pandas as pd
    rx = re.compile(rf'^{root}/([^/]+)/([0-9a-f]+)\.mp4$')
    rows = [{'key': mm.group(2), 'gloss': mm.group(1), 'member': n} for n in names if (mm := rx.match(n))]
    return pd.DataFrame(rows)


def fingerspelling_listing(names: list[str]):
    """One row per image: letter, contributor group (hashed) or clip run, frame number."""
    import pandas as pd
    rows = []
    for name in names:
        p = name.split('/')
        if len(p) != 3 or not p[2].lower().endswith('.jpg'):
            continue
        letter, file = az_lower(p[1]), p[2]
        cid = re.match(r'^ID--(\d+)--', file)
        num = re.search(r'(\d+)\.jpg$', file, re.I)
        rows.append({'file': file, 'member': name, 'letter': letter,
                     'kind': 'dynamic' if letter in DYNAMIC_LETTERS else 'static',
                     'contributor': anon(cid.group(1)) if cid else '',
                     'frame_no': int(num.group(1)) if (num and not cid) else -1})
    df = pd.DataFrame(rows)
    # numbered frames: a gap > RUN_GAP starts a new clip (as in tools/build_poses.py)
    df['run'] = ''
    for letter, g in df[df.contributor == ''].sort_values('frame_no').groupby('letter'):
        run, prev = 0, None
        for idx, n in g['frame_no'].items():
            if prev is not None and n - prev > RUN_GAP:
                run += 1
            df.at[idx, 'run'] = f'{letter}-run{run}'
            prev = n
    df['group'] = df.contributor.where(df.contributor != '', df.run)
    return df

In [ ]:
%%writefile {PKG}/azsl/split.py
"""Signer-independent splits for AzSLD.

AzSLD ships no signer ids and Words_200 clips are UUID-named, which is how the old
I3D repo ended up with the same signers in train and test. Here every Sentences
recording gets a pseudo-signer:

  1. faces — a mean SFace embedding per recording (landmarks.Extractor._embed),
     clustered per RECORDING (average linkage, cosine) at similarity `sim`. Not per
     sitting: 29 of 179 sittings have two people taking turns, and a sitting's average
     resembles neither. The clusters are identical from 0.50 to 0.60 (a clear gap).
  2. split groups — pseudo-signers are tied if any of their sittings look alike
     (split_groups), so a look-alike can never sit on both sides of the split.
  3. one group → split assignment, chosen so the 200-word vocabulary is covered in
     val/test, and reused for every task (words, sentence-cut glosses, sentences),
     so a test signer is never seen in training by any model.

Sessions (recording timestamps, `gap_min`) only define sittings for step 2 and give
recordings without a face their sitting's signer. Real signer ids from the dataset
authors can be passed as an override CSV (rec_id, signer).
"""
import numpy as np


def sessions(recs, gap_min: float = 15.0):
    """recs: DataFrame with rec_id, stamp → Series rec_id → session id."""
    import pandas as pd
    r = recs.dropna(subset=['stamp']).sort_values('stamp')
    sid, prev, out = -1, None, {}
    for rid, t in zip(r.rec_id, r.stamp):
        if prev is None or t.date() != prev.date() or (t - prev).total_seconds() > gap_min * 60:
            sid += 1
        out[rid] = f's{sid:04d}'
        prev = t
    return pd.Series(out, name='session')


def _norm(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)


def cluster_signers(rec_ids, emb, sim: float = 0.55):
    """→ (dict rec_id → signer, report dict). `emb` rows are L2-normalised SFace means."""
    from sklearn.cluster import AgglomerativeClustering
    if len(rec_ids) == 1:
        labels = np.zeros(1, int)
    else:
        labels = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                         distance_threshold=1 - sim).fit_predict(emb)
    signer = {rid: f'p{lab:03d}' for rid, lab in zip(rec_ids, labels)}
    sizes = np.bincount(labels)
    labs = np.unique(labels)
    C = np.stack([_norm(emb[labels == lab].mean(0)) for lab in labs])
    S = C @ C.T
    np.fill_diagonal(S, -1)
    # purity: share of face pairs < 0.4 inside a cluster (two people in one row shows up here)
    rng, mixed = np.random.default_rng(0), []
    for lab in labs[sizes[labs] >= 20]:
        ix = np.where(labels == lab)[0]
        ix = rng.choice(ix, min(400, len(ix)), replace=False)
        P = emb[ix] @ emb[ix].T
        mixed.append(float((P[np.triu_indices(len(ix), 1)] < 0.4).mean()))
    report = {'signers': int(len(labs)), 'tiny_signers_lt20_recordings': int((sizes < 20).sum()),
              'worst_cluster_mixed_pairs': round(max(mixed, default=0.0), 4),
              'nearest_other_signer_sim': [round(float(x), 3) for x in np.sort(S.max(1))[::-1][:10]]}
    return signer, report


def assign_signers(recs, emb_ids, emb, sim=0.55, gap_min=15.0, override=None):
    """recs → DataFrame rec_id, session, signer, has_emb. Recordings without a face fall back to
    their session's signer; with neither they become 'unk-<session>'."""
    ses = sessions(recs, gap_min)
    out = recs[['rec_id']].copy()
    out['session'] = out.rec_id.map(ses).fillna('nosession')
    has = set(emb_ids)
    out['has_emb'] = out.rec_id.isin(has)
    signer, report = ({}, {'signers': 0})
    if len(emb_ids):
        signer, report = cluster_signers(list(emb_ids), _norm(np.asarray(emb, np.float32)), sim)
    out['signer'] = out.rec_id.map(signer)
    maj = out.dropna(subset=['signer']).groupby('session').signer.agg(lambda s: s.value_counts().index[0])
    miss = out.signer.isna()
    out.loc[miss, 'signer'] = out.loc[miss, 'session'].map(maj)
    out['signer'] = out.signer.fillna('unk-' + out.session)
    if override is not None and len(override):
        o = dict(zip(override.rec_id, override.signer.astype(str)))
        out['signer'] = [o.get(r, s) for r, s in zip(out.rec_id, out.signer)]
        report['override'] = len(o)
    # same signer recording the same sentence twice is rare (retakes); many means merged people
    folder = out.rec_id.str.split('/').str[0]
    rep = (out.assign(folder=folder).groupby(['signer', 'folder']).size() > 1).sum()
    report['signer_sentence_repeats'] = int(rep)
    report['recordings'] = len(out)
    report['without_face'] = int((~out.has_emb).sum())
    return out, report


def _units(signers, emb_ids, emb):
    """(session, signer) units — one sitting of one person — and their centroids."""
    sig = dict(zip(signers.rec_id, signers.signer))
    ses = dict(zip(signers.rec_id, signers.session))
    E = np.asarray(emb, np.float32)
    groups = {}
    for i, rid in enumerate(emb_ids):
        if rid in sig:
            groups.setdefault((ses[rid], sig[rid]), []).append(i)
    keys = sorted(groups)
    C = np.stack([_norm(E[groups[k]].mean(0)) for k in keys]) if keys else np.zeros((0, 1))
    return keys, C, {k: len(v) for k, v in groups.items()}


def split_groups(signers, emb_ids, emb, link_sim=0.50):
    """Tie pseudo-signers that share a look-alike sitting: if ANY (sitting, signer) unit of one is
    >= link_sim similar to any unit of the other, they always land in the same split. Units, not
    cluster averages: an average over many recordings hides one look-alike sitting. On AzSLD one
    person's sittings score 0.6+ and different people mostly < 0.5.
    Union-find over the pairs → dict signer → group id."""
    keys, C, _ = _units(signers, emb_ids, emb)
    parent = {n: n for n in signers.signer.unique()}

    def root(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    if len(keys) > 1:
        I, J = np.where(np.triu(C @ C.T, 1) >= link_sim)
        for i, j in zip(I, J):
            if keys[i][1] != keys[j][1]:
                parent[root(keys[i][1])] = root(keys[j][1])
    roots = sorted({root(n) for n in parent})
    gid = {r: f'g{k:02d}' for k, r in enumerate(roots)}
    return {n: gid[root(n)] for n in parent}


def closest_pairs(signers, emb_ids, emb, top=8, cross_split_only=False):
    """Most similar sittings of different pseudo-signers, with their splits — read next to the
    contact sheet. With split_groups() no pair at or above link_sim straddles two splits."""
    keys, C, n = _units(signers, emb_ids, emb)
    split = dict(zip(signers.signer, signers.split))
    S = np.triu(C @ C.T, 1)
    order = np.dstack(np.unravel_index(np.argsort(-S, axis=None), S.shape))[0]
    out = []
    for i, j in order:
        if j <= i or keys[i][1] == keys[j][1]:
            continue
        a, b = keys[i], keys[j]
        if cross_split_only and split.get(a[1]) == split.get(b[1]):
            continue
        out.append({'a': f'{a[1]} {a[0]}', 'b': f'{b[1]} {b[0]}', 'sim': round(float(S[i, j]), 3),
                    'splits': f'{split.get(a[1])}/{split.get(b[1])}', 'recordings': f'{n[a]}/{n[b]}'})
        if len(out) >= top:
            break
    return out


def pick_split(df, group_col: str, label_col: str, fracs=(0.7, 0.15, 0.15), seed: int = 0,
               trials: int = 4000, forced_train=()):
    """Random search over whole groups → dict group → 'train'|'val'|'test'.

    Score: classes (with ≥3 groups) missing from test/val, then distance from target
    sample fractions. Groups in `forced_train` (unknown provenance) never go to val/test.
    """
    rng = np.random.default_rng(seed)
    counts = df.groupby(group_col).size()
    labels_of = df.groupby(group_col)[label_col].agg(set)
    groups = [g for g in counts.index if g not in set(forced_train)]
    n_groups_per_label = df[~df[group_col].isin(forced_train)].groupby(label_col)[group_col].nunique()
    testable = set(n_groups_per_label[n_groups_per_label >= 3].index)
    N = counts[groups].sum()
    best, best_score = None, None
    for _ in range(trials):
        order = rng.permutation(groups)
        assign, n_test, n_val = {}, 0, 0
        for g in order:
            if n_test < fracs[2] * N:
                assign[g], n_test = 'test', n_test + counts[g]
            elif n_val < fracs[1] * N:
                assign[g], n_val = 'val', n_val + counts[g]
            else:
                assign[g] = 'train'
        cov = {s: set().union(*[labels_of[g] for g in assign if assign[g] == s] or [set()]) for s in ('val', 'test')}
        miss_t, miss_v = len(testable - cov['test']), len(testable - cov['val'])
        dev = abs(n_test / N - fracs[2]) + abs(n_val / N - fracs[1])
        score = (miss_t, miss_v, round(float(dev), 3))
        if best_score is None or score < best_score:
            best, best_score = assign, score
    for g in forced_train:
        best[g] = 'train'
    return best, {'missing_test': best_score[0], 'missing_val': best_score[1],
                  'fraction_error': best_score[2], 'testable_classes': len(testable)}


def folds(df, group_col: str, k: int = 5):
    """Train groups → fold 0..k-1, balancing sample counts (largest groups first)."""
    sizes = df.groupby(group_col).size().sort_values(ascending=False)
    load, out = [0] * k, {}
    for g, n in sizes.items():
        f = int(np.argmin(load))
        out[g], load[f] = f, load[f] + n
    return out


def baselines(df, label_col: str, split_col: str = 'split') -> dict:
    """Per split: samples, classes, and what always answering the most frequent train class scores."""
    train = df[df[split_col] == 'train']
    top = train[label_col].value_counts()
    major = top.index[0] if len(top) else None
    out = {'majority_class': major, 'train_classes': int(train[label_col].nunique())}
    for s in ('train', 'val', 'test'):
        d = df[df[split_col] == s]
        out[s] = {
            'samples': int(len(d)),
            'classes': int(d[label_col].nunique()),
            'majority_acc': round(float((d[label_col] == major).mean()), 4) if len(d) else None,
            'chance_balanced_acc': round(1 / max(1, d[label_col].nunique()), 4),
            'unseen_in_train': int(len(set(d[label_col]) - set(train[label_col]))) if s != 'train' else 0,
        }
    return out

In [ ]:
%%writefile {PKG}/azsl/pipeline.py
"""The steps the Kaggle notebooks run. Every step is resumable and takes `limit` for smoke tests
(with `limit`, archives are read member-by-member over HTTP instead of being downloaded)."""
import json
import multiprocessing as mp
import os
import shutil
import tempfile
import time
import zipfile
from collections import deque
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait
from concurrent.futures.process import BrokenProcessPool

from . import landmarks, sources, store, zenodo
from . import split as S

_EX = None
_ZIP = None


# ---------------------------------------------------------------- workers

def _init_worker(paths, face, embed, zip_path):
    global _EX, _ZIP
    # MediaPipe's C++ side logs ~20 warnings per landmarker; thousands of videos would flood the Kaggle log
    os.environ.setdefault('GLOG_minloglevel', '2')
    os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
    import cv2
    cv2.setNumThreads(1)                     # one process per core already
    _EX = landmarks.Extractor(paths, face=face, embed=embed)
    _ZIP = zipfile.ZipFile(zip_path) if zip_path else None


def _read(src):
    return _ZIP.read(src) if isinstance(src, str) else zenodo.read_member(src)


def _video_job(task):
    item_id, src, extra = task
    t0 = time.time()
    data = _read(src)
    fd, tmp = tempfile.mkstemp(suffix='.mp4')
    try:
        with os.fdopen(fd, 'wb') as f:
            f.write(data)
        arrays, meta, face = _EX.video(tmp)
    finally:
        os.remove(tmp)
    meta.update(extra)
    meta['sec'] = round(time.time() - t0, 2)
    return item_id, arrays, meta, face


def _image_job(task):
    item_id, src, extra = task
    arrays, meta = _EX.image(_read(src))
    meta.update(extra)
    return item_id, arrays, meta, None


def run_pool(tasks, job, writer, init_args, workers=None, time_budget_h=11.0, log_every=250, label=''):
    """Runs `job` over tasks [(id, source, extra meta)] in spawned processes, writing as results come.
    Stops taking new tasks after `time_budget_h` so a Kaggle run can finish and save its output.
    A crashed worker (MediaPipe segfault) restarts the pool; its in-flight tasks get one retry."""
    workers = workers or os.cpu_count() or 2
    done = writer.done_ids()
    queue = deque(t for t in tasks if t[0] not in done)
    total = len(queue)
    workers = max(1, min(workers, total))
    print(f'{label}: {len(tasks)} items, {len(done)} already done, {total} to go, {workers} workers', flush=True)
    t0, deadline = time.time(), time.time() + time_budget_h * 3600
    errors, retries, n, frames = [], {}, 0, 0
    ctx = mp.get_context('spawn')
    fruitless = 0                            # pool restarts in a row without a finished item
    while queue and time.time() < deadline:
        pending, n_start = {}, n
        try:
            with ProcessPoolExecutor(workers, mp_context=ctx, initializer=_init_worker, initargs=init_args) as ex:
                while queue or pending:
                    while queue and len(pending) < workers * 3 and time.time() < deadline:
                        t = queue.popleft()
                        try:
                            pending[ex.submit(job, t)] = t
                        except BrokenProcessPool:
                            queue.appendleft(t)      # not in `pending` yet — would be lost
                            raise
                    if not pending:
                        break
                    finished, _ = wait(pending, return_when=FIRST_COMPLETED)
                    for f in finished:
                        t = pending.pop(f)
                        try:
                            item_id, arrays, meta, face = f.result()
                        except BrokenProcessPool:
                            raise
                        except Exception as e:  # noqa: BLE001 — one bad file must not stop the run
                            errors.append({'id': t[0], 'error': repr(e)[:300]})
                            continue
                        writer.add(item_id, arrays, meta, face)
                        n, frames = n + 1, frames + meta['n_frames']
                        if n % log_every == 0:
                            el = time.time() - t0
                            eta = el / n * (total - n) / 3600
                            print(f'  {n}/{total}  {frames / el:.0f} frames/s  {el / 3600:.2f} h  ETA {eta:.2f} h',
                                  flush=True)
        except BrokenProcessPool:
            fruitless = fruitless + 1 if n == n_start else 0
            if fruitless >= 3:
                writer.flush()
                raise RuntimeError('workers keep crashing before finishing anything — mediapipe or a model '
                                   'is broken in this environment (run the self-test cell)')
            print('  worker crashed — restarting pool', flush=True)
            for t in pending.values():
                retries[t[0]] = retries.get(t[0], 0) + 1
                if retries[t[0]] <= 1:
                    queue.appendleft(t)
                else:
                    errors.append({'id': t[0], 'error': 'worker crashed twice'})
    writer.flush()
    if errors:
        import csv
        path = os.path.join(writer.dir, 'errors.csv')
        new = not os.path.exists(path)
        with open(path, 'a', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=['id', 'error'])
            if new:
                w.writeheader()
            w.writerows(errors)
    el = time.time() - t0
    left = total - n - len(errors)
    summary = {'done': n, 'errors': len(errors), 'remaining': left, 'hours': round(el / 3600, 2),
               'frames_per_s': round(frames / el, 1) if el else 0.0, 'complete': left == 0}
    print(f'{label}: {summary}', flush=True)
    if left:
        print(f'  {left} items left (time budget) — rerun with this output attached to resume', flush=True)
    return summary


# ---------------------------------------------------------------- steps

def setup_models(model_dir='/tmp/azsl_models', sface=False):
    return landmarks.ensure_models(model_dir, sface=sface)


def mediapipe_ok(paths):
    """Creates a hand landmarker in a fresh process (a broken mediapipe/numpy pairing can crash
    the interpreter). → mediapipe version, or None."""
    import subprocess
    import sys
    code = ('import mediapipe as mp, numpy as np\n'
            'from mediapipe.tasks.python import vision as V, BaseOptions as B\n'
            f'h = V.HandLandmarker.create_from_options(V.HandLandmarkerOptions(base_options=B(model_asset_path={paths["hand"]!r})))\n'
            'h.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.zeros((64, 64, 3), np.uint8)))\n'
            'print(mp.__version__)')
    r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-1500:])
        return None
    return r.stdout.strip().splitlines()[-1]


def run_index(out_dir, folders=None):
    """Sentences annotations → index/tags.csv, recordings.csv, sentences.csv (a few MB, ~minutes)."""
    members = zenodo.list_members(zenodo.archive_url('sentences'))
    if folders is not None:
        keep = {str(f) for f in folders}
        members = [m for m in members if m.name.split('/')[1] in keep or m.name.endswith('sentence_list.csv')]
    tags, recs, sents = sources.sentences_index(members)
    d = os.path.join(out_dir, 'index')
    os.makedirs(d, exist_ok=True)
    tags.to_csv(os.path.join(d, 'tags.csv'), index=False)
    recs.to_csv(os.path.join(d, 'recordings.csv'), index=False)
    sents.to_csv(os.path.join(d, 'sentences.csv'), index=False)
    print(f'index: {len(recs)} recordings ({int(recs.cam2.sum())} with Cam2), {len(tags)} tags '
          f'({int(tags.dup.sum())} duplicates), {tags.gloss.nunique()} glosses, {len(sents)} sentences')
    return tags, recs, sents


def _archive(key, tmp_dir, limit):
    """Full runs download the archive (one big request is kinder to Zenodo than thousands);
    smoke tests read the needed members over HTTP."""
    url = zenodo.archive_url(key)
    if limit:
        return None, zenodo.list_members(url)
    path = os.path.join(tmp_dir, zenodo.ARCHIVES[key])
    t0 = time.time()
    zenodo.download(url, path)
    print(f'{zenodo.ARCHIVES[key]}: {os.path.getsize(path) / 1e9:.2f} GB in {time.time() - t0:.0f} s', flush=True)
    with zipfile.ZipFile(path) as z:
        names = [i.filename for i in z.infolist() if not i.is_dir()]
    return path, names


def _by_name(members):
    """name → task source: the name itself (read from the local zip) or a zenodo.Member (HTTP)."""
    return {m if isinstance(m, str) else m.name: m for m in members}


def run_words200(out_dir, paths, workers=None, limit=None, tmp_dir='/tmp', time_budget_h=11.0):
    zpath, members = _archive('words200', tmp_dir, limit)
    by_name = _by_name(members)
    listing = sources.words_listing(list(by_name))
    if limit:
        listing = listing.groupby('gloss').head(1).head(limit)
    tasks = [(r.key, by_name[r.member], {'key': r.key, 'gloss': r.gloss}) for r in listing.itertuples()]
    writer = store.PartWriter(os.path.join(out_dir, 'landmarks', 'words200'), items_per_part=1000)
    return run_pool(tasks, _video_job, writer, (paths, True, False, zpath), workers, time_budget_h,
                    label='Words_200')


def run_fingerspelling(out_dir, paths, workers=None, limit=None, tmp_dir='/tmp', time_budget_h=11.0):
    zpath, members = _archive('fingerspelling', tmp_dir, limit)
    by_name = _by_name(members)
    listing = sources.fingerspelling_listing(list(by_name))
    if limit:
        listing = listing.groupby('letter').head(1).head(limit)
    tasks = [(r.member.split('/', 1)[1], by_name[r.member],
              {'letter': r.letter, 'kind': r.kind, 'group': r.group, 'run': r.run, 'frame_no': r.frame_no})
             for r in listing.itertuples()]
    writer = store.PartWriter(os.path.join(out_dir, 'landmarks', 'fingerspelling'), items_per_part=4000)
    return run_pool(tasks, _image_job, writer, (paths, True, False, zpath), workers, time_budget_h,
                    log_every=1000, label='Fingerspelling')


def run_sentences(out_dir, paths, shard=0, num_shards=1, camera='Cam2', workers=None, limit=None,
                  time_budget_h=11.0, resume_from=()):
    """One shard of the Sentences recordings, read one by one from the 43.9 GB zip.
    Cam2 (frontal) also stores a face embedding per recording for the signer split."""
    members = zenodo.list_members(zenodo.archive_url('sentences'))
    recs = sources.sentence_recordings(members, camera)
    mine = [m for k, m in enumerate(recs) if k % num_shards == shard]
    if limit:
        mine = mine[:limit]
    sub = os.path.join(out_dir, 'landmarks', f'sentences_{camera.lower()}', f'shard{shard:02d}')
    for prev in resume_from:                 # earlier run of this shard attached as input
        if os.path.exists(os.path.join(prev, 'manifest.csv')) and not os.path.exists(os.path.join(sub, 'manifest.csv')):
            shutil.copytree(prev, sub, dirs_exist_ok=True)
            print(f'resumed from {prev}', flush=True)
    embed = camera == 'Cam2'
    writer = store.PartWriter(sub, items_per_part=200, embeddings=embed)
    tasks = [(sources.rec_id(m.name), m, {'rec_id': sources.rec_id(m.name), 'camera': camera, 'bytes': m.size})
             for m in mine]
    return run_pool(tasks, _video_job, writer, (paths, True, embed, None), workers, time_budget_h,
                    log_every=100, label=f'Sentences {camera} shard {shard}/{num_shards}')


# ---------------------------------------------------------------- split

def run_split(index_dir, words_dirs, fs_dirs, sentence_dirs, out_dir, sim=0.55, link_sim=0.50, gap_min=15.0,
              seed=0, override_csv=None, heldout_sentence_frac=0.1, min_clips=20, k_folds=5, trials=4000):
    """Pseudo-signers → one signer-level train/val/test split → per-task split files + benchmark.json."""
    import numpy as np
    import pandas as pd
    tags = pd.read_csv(os.path.join(index_dir, 'tags.csv'), dtype={'key': str, 'rec_id': str})
    recs = pd.read_csv(os.path.join(index_dir, 'recordings.csv'), dtype={'rec_id': str, 'folder': str},
                       parse_dates=['stamp'])
    sents = pd.read_csv(os.path.join(index_dir, 'sentences.csv'), dtype={'folder': str})

    sen = store.LandmarkStore(sentence_dirs)
    emb_ids, emb, thumbs = sen.embeddings()
    ok = ~np.isnan(emb).any(1) if len(emb) else np.zeros(0, bool)
    emb_ids, emb = [i for i, k in zip(emb_ids, ok) if k], emb[ok]
    # compact copy for tuning the clustering offline (anonymous vectors, no images; never committed)
    priv = os.path.join(os.path.dirname(os.path.abspath(out_dir)), 'private')
    os.makedirs(priv, exist_ok=True)
    np.savez_compressed(os.path.join(priv, 'face_embeddings.npz'), ids=np.array(emb_ids), emb=emb.astype(np.float16))
    override = pd.read_csv(override_csv, dtype=str) if override_csv else None
    signers, rep = S.assign_signers(recs, emb_ids, emb, sim=sim, gap_min=gap_min, override=override)
    group_of = S.split_groups(signers, emb_ids, emb, link_sim)
    signers['split_group'] = signers.signer.map(group_of)
    m = sen.manifest.set_index('rec_id') if 'rec_id' in sen.manifest else sen.manifest.set_index('id')
    signers['has_landmarks'] = signers.rec_id.isin(m.index)
    signers['frontal'] = signers.rec_id.map(m['frontal'])
    signer_of = dict(zip(signers.rec_id, signers.signer))

    words = store.LandmarkStore(words_dirs).manifest[['id', 'gloss', 'hand_rate', 'n_frames']].rename(columns={'id': 'key'})
    vocab = set(words.gloss)
    t = tags[~tags.dup].copy()
    t['signer'] = t.rec_id.map(signer_of)
    t = t.dropna(subset=['signer'])
    t['split_group'] = t.signer.map(group_of)

    # one split over split groups (pseudo-signers + the ones ambiguously alike), optimised for
    # covering the 200-word vocabulary in val and test, then shared by every task
    t200 = t[t.gloss.isin(vocab)]
    unknown = sorted({group_of[g] for g in t.signer.unique() if g.startswith('unk-')})
    assign, cover = S.pick_split(t200, 'split_group', 'gloss', seed=seed, trials=trials, forced_train=unknown)
    for g in set(group_of.values()):
        assign.setdefault(g, 'train')
    fold_of = S.folds(t200[t200.split_group.map(assign) == 'train'], 'split_group', k_folds)
    signers['split'] = signers.split_group.map(assign).fillna('train')
    signers['fold'] = signers.split_group.map(fold_of)
    split_of, fold_by_signer = dict(zip(signers.signer, signers.split)), dict(zip(signers.signer, signers.fold))

    # task 1 — Words_200 clips (link via tag key → recording → signer)
    w = words.merge(tags[['key', 'rec_id', 'start', 'end']], on='key', how='left')
    w['signer'] = w.rec_id.map(signer_of).fillna('unknown')
    w['split'] = w.signer.map(split_of).fillna('train')
    w['fold'] = w.signer.map(fold_by_signer)
    # task 2 — every annotated gloss cut from the sentences (slice the recording's landmarks)
    n_clips, n_sig = t.groupby('gloss').size(), t.groupby('gloss').signer.nunique()
    t['in_w200'] = t.gloss.isin(vocab)
    t['min_clips'] = t.gloss.map((n_clips >= min_clips) & (n_sig >= 3))
    t['split'], t['fold'] = t.signer.map(split_of), t.signer.map(fold_by_signer)
    t['has_landmarks'] = t.rec_id.isin(m.index)
    # task 3 — whole sentences; a held-out set of sentence folders measures generalisation to unseen sentences
    rng = np.random.default_rng(seed)
    folders = sorted(recs.folder.unique(), key=lambda x: (len(x), x))
    held = set(rng.choice(folders, max(1, int(len(folders) * heldout_sentence_frac)), replace=False))
    s = signers.merge(recs[['rec_id', 'folder', 'frames']], on='rec_id', how='left')
    s['heldout_sentence'] = s.folder.isin(held)
    s = s.merge(sents, on='folder', how='left')      # sentence text + gloss sequence, for Phase 4
    # task 4 — fingerspelling (contributor id or clip run as the group)
    fs = store.LandmarkStore(fs_dirs).manifest
    fs_assign, fs_cover = {}, {}
    for kind, g in fs.groupby('kind'):
        a, c = S.pick_split(g, 'group', 'letter', seed=seed, trials=trials)
        fs_assign.update(a)
        fs_cover[kind] = c
    fs['split'] = fs.group.map(fs_assign)

    os.makedirs(out_dir, exist_ok=True)
    signers.to_csv(os.path.join(out_dir, 'signers.csv'), index=False)
    w.to_csv(os.path.join(out_dir, 'words200_split.csv'), index=False)
    t.to_csv(os.path.join(out_dir, 'glosses_split.csv'), index=False)
    s.to_csv(os.path.join(out_dir, 'sentences_split.csv'), index=False)
    fs[['id', 'letter', 'kind', 'group', 'run', 'frame_no', 'hand_rate', 'split']].to_csv(
        os.path.join(out_dir, 'fingerspelling_split.csv'), index=False)

    split_signers = signers.drop_duplicates('signer').groupby('split').size().to_dict()
    rep['closest_pairs'] = S.closest_pairs(signers, emb_ids, emb)
    cross = [p['sim'] for p in S.closest_pairs(signers, emb_ids, emb, top=1, cross_split_only=True)]
    gsize = signers.groupby('split_group').size()
    rep['split_groups'] = {'count': int(len(gsize)), 'largest_share': round(float(gsize.max() / gsize.sum()), 3),
                           'per_split': signers.drop_duplicates('split_group').groupby('split').size().to_dict(),
                           'max_cross_split_sim': max(cross, default=None), 'link_sim': link_sim}
    bench = {
        'protocol': 'signer-independent; one split-group→split map shared by all tasks; seed %d' % seed,
        'signers': {**rep, 'per_split': split_signers, 'sim_threshold': sim, 'session_gap_min': gap_min},
        'frontal_outliers': int((signers.frontal > 0.35).sum()),
        'words200': {'coverage': cover, 'unlinked_clips': int((w.signer == 'unknown').sum()),
                     **S.baselines(w, 'gloss')},
        'glosses_min%d' % min_clips: S.baselines(t[t.min_clips], 'gloss'),
        'glosses_all': S.baselines(t, 'gloss'),
        'sentences': {'heldout_folders': len(held),
                      **{k: int(v) for k, v in s.groupby('split').size().items()}},
        'fingerspelling': {kind: {'coverage': fs_cover[kind], **S.baselines(g, 'letter')}
                           for kind, g in fs.groupby('kind')},
    }
    with open(os.path.join(out_dir, 'benchmark.json'), 'w', encoding='utf-8') as f:
        json.dump(bench, f, ensure_ascii=False, indent=1, default=str)
    return bench, signers, dict(zip(emb_ids, thumbs))


def signer_sheet(signers, thumbs, per_signer=6, size=72):
    """Contact sheet: one row per pseudo-signer, a few face crops each — to eyeball the clustering.
    Shown in the notebook only; not written to the output."""
    import cv2
    import numpy as np
    rows = []
    for sig, g in signers[signers.rec_id.isin(thumbs)].groupby('signer'):
        tiles = []
        for rid in g.rec_id.sample(min(per_signer, len(g)), random_state=0):
            im = cv2.imdecode(np.frombuffer(thumbs[rid], np.uint8), cv2.IMREAD_COLOR)
            if im is not None:
                tiles.append(cv2.resize(im, (size, size)))
        tiles += [np.zeros((size, size, 3), np.uint8)] * (per_signer - len(tiles))
        label = np.full((size, 90, 3), 255, np.uint8)
        cv2.putText(label, f'{sig} {len(g)}', (2, size // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1)
        rows.append(np.hstack([label] + tiles))
    return cv2.cvtColor(np.vstack(rows), cv2.COLOR_BGR2RGB) if rows else None

In [ ]:
%%writefile {PKG}/azsl/features.py
"""Landmarks → per-frame model input (numpy only, so the browser can mirror it in JS).

Raw (from the extraction, see landmarks.py), reduced to what the model reads:
  hand        T×2×21×3   image coords (slot 0 = signer's right hand)
  hand_world  T×2×21×3   metric hand shape
  pose        T×9×3      POSE_IDX: nose, shoulders, elbows, wrists, hips (x, y, z — only x, y are used)
  face        T×102×2    landmarks.FACE_IDX, x, y
  blend       T×52       face blendshapes

Features per frame (N_FEAT = 656), all body-relative: centre = mid-shoulders, unit = shoulder
width (medians over the clip), x first multiplied by width/height so geometry is square:
  hand positions 126 · hand shape (world, ×10) 126 · hand velocity 126 · pose xy 18 ·
  face xy 204 · blendshapes 52 · presence flags 4 (right hand, left hand, face, pose)
Missing values are 0 with the presence flag telling the model.

Augmentation (training only) works on the raw coordinates: temporal crop/jitter around the
sign, speed change, frame drop, horizontal flip (hands, pose, face points and blendshapes
swap sides — a mirrored sign is the same sign), small rotation/scale/shear, hand dropout.
"""
import numpy as np

from .landmarks import FACE_IDX

# Bump on ANY change to reduce(), to_square() or features(): models, the browser port and the
# landmark server compare it and refuse a mismatch.
FEATURE_VERSION = 1

POSE_IDX = [0, 11, 12, 13, 14, 15, 16, 23, 24]
POSE_FLIP = [0, 2, 1, 4, 3, 6, 5, 8, 7]
N_FEAT = 126 * 3 + 18 + 204 + 52 + 4
MAX_LEN = 64
TRAIN_FPS = 30          # the training landmarks: every frame of 30 fps video (stride 1)
# column ranges of features(), for ablations (train cfg 'groups')
GROUPS = {'hands': (0, 126), 'hand_shape': (126, 252), 'hand_vel': (252, 378), 'pose': (378, 396),
          'face': (396, 600), 'blend': (600, 652), 'flags': (652, 656)}
assert GROUPS['flags'][1] == N_FEAT


def columns(groups=None) -> np.ndarray:
    """Feature columns of the named groups (None = all)."""
    if not groups:
        return np.arange(N_FEAT)
    return np.concatenate([np.arange(*GROUPS[g]) for g in groups])

BLEND_NAMES = [
    '_neutral', 'browDownLeft', 'browDownRight', 'browInnerUp', 'browOuterUpLeft', 'browOuterUpRight',
    'cheekPuff', 'cheekSquintLeft', 'cheekSquintRight', 'eyeBlinkLeft', 'eyeBlinkRight', 'eyeLookDownLeft',
    'eyeLookDownRight', 'eyeLookInLeft', 'eyeLookInRight', 'eyeLookOutLeft', 'eyeLookOutRight', 'eyeLookUpLeft',
    'eyeLookUpRight', 'eyeSquintLeft', 'eyeSquintRight', 'eyeWideLeft', 'eyeWideRight', 'jawForward', 'jawLeft',
    'jawOpen', 'jawRight', 'mouthClose', 'mouthDimpleLeft', 'mouthDimpleRight', 'mouthFrownLeft',
    'mouthFrownRight', 'mouthFunnel', 'mouthLeft', 'mouthLowerDownLeft', 'mouthLowerDownRight', 'mouthPressLeft',
    'mouthPressRight', 'mouthPucker', 'mouthRight', 'mouthRollLower', 'mouthRollUpper', 'mouthShrugLower',
    'mouthShrugUpper', 'mouthSmileLeft', 'mouthSmileRight', 'mouthStretchLeft', 'mouthStretchRight',
    'mouthUpperUpLeft', 'mouthUpperUpRight', 'noseSneerLeft', 'noseSneerRight',
]   # MediaPipe face_landmarker order (checked against a live detection)


def _swap_lr(name):
    for a, b in (('Left', 'Right'), ('Right', 'Left')):
        if name.endswith(a):
            return name[:-len(a)] + b
    return name


BLEND_FLIP = [BLEND_NAMES.index(_swap_lr(n)) for n in BLEND_NAMES]

# face-mesh mirror pairs (topology of the lip/eye/brow contours; brows, nose and outer lips
# also checked on a flipped frame)
_MIRROR = {
    61: 291, 185: 409, 40: 270, 39: 269, 37: 267, 146: 375, 91: 321, 181: 405, 84: 314,
    78: 308, 191: 415, 80: 310, 81: 311, 82: 312, 95: 324, 88: 318, 178: 402, 87: 317,
    263: 33, 249: 7, 390: 163, 373: 144, 374: 145, 380: 153, 381: 154, 382: 155, 362: 133,
    466: 246, 388: 161, 387: 160, 386: 159, 385: 158, 384: 157, 398: 173,
    276: 46, 283: 53, 282: 52, 295: 65, 285: 55, 300: 70, 293: 63, 334: 105, 296: 66, 336: 107,
    98: 327, 234: 454, 468: 473,
}
_MIRROR.update({b: a for a, b in list(_MIRROR.items())})
FACE_FLIP = [FACE_IDX.index(_MIRROR.get(i, i)) for i in FACE_IDX]
assert sorted(FACE_FLIP) == list(range(len(FACE_IDX))) and sorted(BLEND_FLIP) == list(range(52))


def reduce(arrays: dict) -> dict:
    """Store arrays (landmarks.FIELDS) → the compact raw dict the model needs (float16)."""
    return {
        'hand': arrays['hand'].astype(np.float16),
        'hand_world': arrays['hand_world'].astype(np.float16),
        'pose': arrays['pose'][:, POSE_IDX, :3].astype(np.float16),
        'face': arrays['face'][..., :2].astype(np.float16),
        'blend': arrays['blend'].astype(np.float16),
    }


def _affine(rng, strength=1.0):
    a = np.deg2rad(rng.uniform(-10, 10)) * strength
    s = 1 + rng.uniform(-0.15, 0.15) * strength
    sh = rng.uniform(-0.12, 0.12) * strength
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    M = R @ np.array([[1, sh], [0, 1]]) * s
    return M, a


def augment(raw: dict, rng, aspect: float, core=None) -> dict:
    """raw: square-space dict (to_square), `core` = (lo, hi) frame range of the sign inside it."""
    T = raw['hand'].shape[0]
    lo, hi = core if core is not None else (0, T)
    # temporal crop: jitter each boundary by up to ±4 frames inside the stored margin
    lo = int(np.clip(lo + rng.integers(-4, 5), 0, max(0, hi - 2)))
    hi = int(np.clip(hi + rng.integers(-4, 5), lo + 2, T))
    idx = np.arange(lo, hi)
    # speed 0.8–1.25× by resampling indices; drop up to 10% of frames
    speed = np.exp(rng.uniform(np.log(0.8), np.log(1.25)))
    n = max(2, int(round(len(idx) / speed)))
    idx = idx[np.clip(np.round(np.linspace(0, len(idx) - 1, n)).astype(int), 0, len(idx) - 1)]
    if len(idx) > 8 and rng.random() < 0.5:
        keep = rng.random(len(idx)) > rng.uniform(0, 0.1)
        idx = idx[keep] if keep.sum() >= 2 else idx
    out = {k: v[idx].astype(np.float32) for k, v in raw.items()}

    if rng.random() < 0.5:                     # mirror: the same sign signed with the other hand
        out = flip(out, aspect)

    # rotation / scale / shear about the shoulder centre, in square (aspect-corrected) space
    M, ang = _affine(rng)
    c = _centre(out)
    for k in ('hand', 'pose', 'face'):
        xy = out[k][..., :2]
        out[k][..., :2] = (xy - c) @ M.T + c + rng.normal(0, 0.003, xy.shape)
    cw, sw = np.cos(ang), np.sin(ang)
    hw = out['hand_world'][..., :2].copy()
    out['hand_world'][..., 0] = hw[..., 0] * cw - hw[..., 1] * sw
    out['hand_world'][..., 1] = hw[..., 0] * sw + hw[..., 1] * cw

    # a hand missing for a stretch (detector misses happen live too)
    for s in (0, 1):
        if rng.random() < 0.15:
            L = len(idx)
            span = max(1, int(L * rng.uniform(0.1, 0.4)))
            st = rng.integers(0, max(1, L - span + 1))
            out['hand'][st:st + span, s] = np.nan
            out['hand_world'][st:st + span, s] = np.nan
    return out


def flip(raw: dict, aspect: float) -> dict:
    """Mirror a square-space dict (x ∈ [0, aspect])."""
    out = {k: v.copy() for k, v in raw.items()}
    W = aspect
    out['hand'] = out['hand'][:, ::-1].copy()
    out['hand_world'] = out['hand_world'][:, ::-1].copy()
    out['hand'][..., 0] = W - out['hand'][..., 0]
    out['hand_world'][..., 0] = -out['hand_world'][..., 0]
    out['pose'] = out['pose'][:, POSE_FLIP]
    out['pose'][..., 0] = W - out['pose'][..., 0]
    out['face'] = out['face'][:, FACE_FLIP]
    out['face'][..., 0] = W - out['face'][..., 0]
    out['blend'] = out['blend'][:, BLEND_FLIP]
    return out


def to_square(raw: dict, aspect: float) -> dict:
    """Image x ∈ [0,1] → x·width/height, so distances are the same in x and y."""
    out = {k: v.astype(np.float32) for k, v in raw.items()}
    for k in ('hand', 'pose', 'face'):
        out[k][..., 0] *= aspect
    return out


def _centre(d):
    sh = d['pose'][:, 1:3, :2]
    c = np.nanmedian(sh.reshape(-1, 2), axis=0) if np.isfinite(sh).any() else np.array([np.nan, np.nan])
    if not np.isfinite(c).all():
        pts = d['hand'][..., :2].reshape(-1, 2)
        c = np.nanmean(pts, axis=0) if np.isfinite(pts).any() else np.array([0.5, 0.5])
    return c.astype(np.float32)


def _scale(d):
    sh = d['pose'][:, 1:3, :2]
    w = np.linalg.norm(sh[:, 0] - sh[:, 1], axis=-1)
    w = np.nanmedian(w) if np.isfinite(w).any() else np.nan
    return float(w) if np.isfinite(w) and w > 1e-3 else 0.3


def features(d: dict) -> np.ndarray:
    """Square-space compact dict → T×N_FEAT float32 (at most MAX_LEN frames, uniformly resampled)."""
    T = d['hand'].shape[0]
    if T > MAX_LEN:
        sel = np.round(np.linspace(0, T - 1, MAX_LEN)).astype(int)
        d = {k: v[sel] for k, v in d.items()}
        T = MAX_LEN
    c, s = _centre(d), _scale(d)
    hand = d['hand'].copy()
    hand[..., :2] = (hand[..., :2] - c) / s
    hand[..., 2] = hand[..., 2] / s
    vel = np.zeros_like(hand)
    vel[1:] = hand[1:] - hand[:-1]
    pose = (d['pose'][..., :2] - c) / s
    face = (d['face'] - c) / s
    flags = np.stack([np.isfinite(d['hand'][:, 0, 0, 0]), np.isfinite(d['hand'][:, 1, 0, 0]),
                      np.isfinite(d['face'][:, 0, 0]), np.isfinite(d['pose'][:, 1, 0])], 1).astype(np.float32)
    x = np.concatenate([hand.reshape(T, -1), (d['hand_world'] * 10).reshape(T, -1), vel.reshape(T, -1),
                        pose.reshape(T, -1), face.reshape(T, -1), d['blend'].reshape(T, -1), flags], 1)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(x, -10, 10).astype(np.float32)


# ---------------------------------------------------------------- the input contract

def drop_face(d: dict) -> dict:
    """Privacy mode / modality dropout: no face points, no blendshapes (the face flag becomes 0)."""
    out = dict(d)
    out['face'] = np.full_like(d['face'], np.nan)
    out['blend'] = np.full_like(d['blend'], np.nan)
    return out


def preprocess(raw: dict, aspect: float, groups=None, face: bool = True) -> np.ndarray:
    """THE entry point from landmarks to model input, the same for training evaluation, the browser
    (ported to JS) and the landmark server.

    raw:    compact dict (reduce() layout: hand, hand_world, pose, face, blend), float, NaN = missing.
            Without face points (privacy mode) pass face=False, or NaN face/blend arrays.
    aspect: frame width / height of the camera the landmarks come from.
    → T'×F float32, T' = min(T, MAX_LEN), F = len(columns(groups)).
    """
    d = to_square(raw, aspect)
    if not face:
        d = drop_face(d)
    x = features(d)
    return x if not groups else x[:, columns(groups)]


def spec() -> dict:
    """Machine-readable description of the contract, written into every model manifest."""
    return {
        'feature_version': FEATURE_VERSION,
        'raw': {'hand': [2, 21, 3], 'hand_world': [2, 21, 3], 'pose': [len(POSE_IDX), 3],
                'face': [len(FACE_IDX), 2], 'blend': [len(BLEND_NAMES)]},
        'raw_notes': ('per frame; image coords x, y in [0, 1] of the frame (x NOT pre-multiplied by '
                      'width/height), hand slot 0 = signer\'s right hand (pose wrists decide), '
                      'pose = MediaPipe pose landmarks POSE_IDX with (x, y, z; only x, y are used), '
                      'face = face-mesh FACE_IDX (x, y), blend = face blendshapes in BLEND_NAMES order, '
                      'NaN = not detected'),
        'pose_idx': POSE_IDX, 'face_idx': list(FACE_IDX), 'blend_names': BLEND_NAMES,
        'n_features': N_FEAT, 'groups': {k: list(v) for k, v in GROUPS.items()},
        'max_len': MAX_LEN, 'resample': 'clips longer than max_len are uniformly resampled to max_len frames',
        'train_fps': TRAIN_FPS,
        'normalization': ('x *= width/height; centre = median mid-shoulder, unit = median shoulder width '
                          '(fallbacks: mean hand point, 0.3); hand xy, z, velocity, pose xy, face xy are '
                          '(p - centre) / unit; hand_world * 10; NaN → 0; clip to ±10'),
    }

In [ ]:
%%writefile {PKG}/azsl/train.py
"""Phase 2: isolated-sign recognition on landmarks (PyTorch), signer-independent.

Data
  train   every annotated sign of the 200-word vocabulary cut from the sentence recordings of
          train signers, with a margin of real context around it (augment() crops inside it)
  val     reported every epoch: the Words_200 clips of val signers — the benchmark — and the
          sentence-cut signs of val signers (a bigger, steadier estimate)
  test    scored once, with the best-on-val checkpoint
Model     Conv1D (depthwise, ECA) + Transformer blocks, as in the top Kaggle isolated-sign
          solutions; ~1.7 M parameters, exported to ONNX for the browser.
Classes are very uneven (MƏN ≈ 30% of Words_200), so batches are drawn with weights n_c^-0.5
and checkpoints are chosen by val macro-F1; plain accuracy is always shown next to the
majority-class baseline.
"""
import csv
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from . import features as FT
from .store import LandmarkStore

KEYS = ('hand', 'hand_world', 'pose', 'face', 'blend')
DEFAULTS = dict(epochs=60, batch=64, lr=1e-3, weight_decay=0.05, warmup=3, label_smoothing=0.1,
                balance_power=0.5, dim=192, stages=2, convs=3, kernel=17, heads=4, drop=0.2,
                head_drop=0.4, margin=8, workers=4, seed=0, groups=None,
                face_drop=0.0,          # P(train clip loses face + blendshapes): one model for both privacy modes
                fold=None, split_id=None)


# ---------------------------------------------------------------- data

class Bank:
    """Compact raw landmarks of many samples: one concatenated array per field + an index."""

    def __init__(self):
        self.chunks, self.rows, self.n = {k: [] for k in KEYS}, [], 0

    def add(self, raw, label, aspect, core, sample_id):
        T = raw['hand'].shape[0]
        for k in KEYS:
            self.chunks[k].append(raw[k])
        self.rows.append((self.n, T, int(label), float(aspect), int(core[0]), int(core[1]), sample_id))
        self.n += T

    def done(self):
        self.data = {k: np.concatenate(v) if v else np.zeros((0,), np.float16) for k, v in self.chunks.items()}
        self.chunks = None
        self.labels = np.array([r[2] for r in self.rows], np.int64)
        self.ids = [r[6] for r in self.rows]
        return self

    def __len__(self):
        return len(self.rows)

    def raw(self, i):
        off, T, label, aspect, lo, hi, _ = self.rows[i]
        return {k: self.data[k][off:off + T] for k in KEYS}, label, aspect, (lo, hi)


def load_sentence_signs(sentence_dirs, tags, vocab, margin=8, log=print):
    """tags (glosses_split.csv rows) → Bank per split, sliced from whole-recording landmarks."""
    store = LandmarkStore(sentence_dirs)
    idx = store.index
    n0 = len(tags)
    tags = tags[tags.rec_id.isin(idx.index) & tags.gloss.isin(vocab) & tags.start.notna() & tags.end.notna()].copy()
    if len(tags) < n0:
        log(f'  {n0 - len(tags)} signs dropped (no landmarks, outside the vocabulary or no frame range)')
    tags['_order'] = list(zip(tags.rec_id.map(idx['dir']), tags.rec_id.map(idx['part'])))
    banks = {s: Bank() for s in ('train', 'val', 'test')}
    t0, skipped = time.time(), 0
    for n, (rid, g) in enumerate(tags.sort_values(['_order', 'rec_id']).groupby('rec_id', sort=False)):
        raw = FT.reduce(store.get(rid, dtype=np.float16))
        T = raw['hand'].shape[0]
        aspect = float(idx.at[rid, 'width']) / float(idx.at[rid, 'height'])
        for r in g.itertuples():
            s0, e0 = int(r.start) - 1, min(int(r.end), T)        # 1-based inclusive → [s0, e0)
            if s0 < 0 or e0 - s0 < 2:
                skipped += 1
                continue
            lo, hi = max(0, s0 - margin), min(T, e0 + margin)
            banks[r.split].add({k: v[lo:hi] for k, v in raw.items()}, vocab[r.gloss], aspect,
                               (s0 - lo, e0 - lo), r.key)
        if n and n % 2000 == 0:
            log(f'  {n} recordings sliced ({time.time() - t0:.0f} s)')
    log('sentence signs: ' + ', '.join(f'{s} {len(b)}' for s, b in banks.items()) + f' (skipped {skipped})')
    return {s: b.done() for s, b in banks.items()}


def load_words200(words_dirs, wsplit, vocab, log=print):
    store = LandmarkStore(words_dirs)
    idx = store.index
    w = wsplit[wsplit.key.isin(idx.index)].copy()
    w['_order'] = list(zip(w.key.map(idx['dir']), w.key.map(idx['part'])))
    banks = {s: Bank() for s in ('train', 'val', 'test')}
    for r in w.sort_values('_order').itertuples():
        raw = FT.reduce(store.get(r.key, dtype=np.float16))
        T = raw['hand'].shape[0]
        if T < 2:
            continue
        aspect = float(idx.at[r.key, 'width']) / float(idx.at[r.key, 'height'])
        banks[r.split].add(raw, vocab[r.gloss], aspect, (0, T), r.key)
    log('Words_200 clips: ' + ', '.join(f'{s} {len(b)}' for s, b in banks.items()))
    return {s: b.done() for s, b in banks.items()}


class SignDataset(torch.utils.data.Dataset):
    """Evaluation goes through features.preprocess() — the exact code the landmark server runs and the
    browser port mirrors. Training adds augmentation (and face dropout) on the same functions."""

    def __init__(self, bank, train=False, flip=False, groups=None, face=True, face_drop=0.0):
        self.bank, self.train, self.flip, self.groups, self.face, self.face_drop = bank, train, flip, groups, face, face_drop
        self.cols = FT.columns(groups) if groups else None

    def __len__(self):
        return len(self.bank)

    def __getitem__(self, i):
        raw, label, aspect, core = self.bank.raw(i)
        lo, hi = core
        if not self.train and not self.flip:
            x = FT.preprocess({k: v[lo:hi] for k, v in raw.items()}, aspect, self.groups, self.face)
            return torch.from_numpy(x), label
        sq = FT.to_square(raw, aspect)
        if self.train:
            rng = np.random.default_rng()
            d = FT.augment(sq, rng, aspect, core)
            if self.face_drop and rng.random() < self.face_drop:
                d = FT.drop_face(d)
        else:                                     # mirror TTA
            d = FT.flip({k: v[lo:hi] for k, v in sq.items()}, aspect)
            if not self.face:
                d = FT.drop_face(d)
        x = FT.features(d)
        return torch.from_numpy(x if self.cols is None else x[:, self.cols]), label


def collate(batch):
    T = max(x.shape[0] for x, _ in batch)
    X = torch.zeros(len(batch), T, batch[0][0].shape[1])
    M = torch.zeros(len(batch), T, dtype=torch.bool)
    for i, (x, _) in enumerate(batch):
        X[i, :len(x)] = x
        M[i, :len(x)] = True
    return X, M, torch.tensor([y for _, y in batch])


# ---------------------------------------------------------------- model

class ECA(nn.Module):
    """Efficient channel attention over the masked time average."""

    def __init__(self, k=5):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)

    def forward(self, x, m):                    # x B×C×T, m B×1×T
        y = (x * m).sum(-1) / m.sum(-1).clamp(min=1)
        return x * torch.sigmoid(self.conv(y[:, None, :]))[:, 0, :, None]


class ConvBlock(nn.Module):
    def __init__(self, dim, k=17, expand=2, drop=0.2):
        super().__init__()
        h = dim * expand
        self.norm, self.pw1 = nn.LayerNorm(dim), nn.Linear(dim, h)
        self.dw = nn.Conv1d(h, h, k, padding=k // 2, groups=h)
        self.bn, self.eca, self.pw2, self.drop = nn.BatchNorm1d(h), ECA(), nn.Linear(h, dim), nn.Dropout(drop)

    def forward(self, x, valid):
        m = valid[:, :, None].to(x.dtype)
        y = F.silu(self.pw1(self.norm(x))) * m
        y = y.transpose(1, 2)
        y = self.eca(F.silu(self.bn(self.dw(y))), m.transpose(1, 2))
        return x + self.drop(self.pw2(y.transpose(1, 2))) * m


class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4, drop=0.2):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(dim, dim * 2), nn.SiLU(), nn.Dropout(drop), nn.Linear(dim * 2, dim))
        self.drop = nn.Dropout(drop)

    def forward(self, x, valid):
        h = self.n1(x)
        a, _ = self.attn(h, h, h, key_padding_mask=~valid, need_weights=False)
        x = x + self.drop(a)
        return x + self.drop(self.ff(self.n2(x)))


class Net(nn.Module):
    def __init__(self, n_feat, n_cls, dim=192, stages=2, convs=3, kernel=17, heads=4, drop=0.2, head_drop=0.4):
        super().__init__()
        self.stem, self.stem_norm = nn.Linear(n_feat, dim), nn.LayerNorm(dim)
        blocks = []
        for _ in range(stages):
            blocks += [ConvBlock(dim, kernel, drop=drop) for _ in range(convs)] + [TransformerBlock(dim, heads, drop)]
        self.blocks = nn.ModuleList(blocks)
        self.norm, self.drop, self.head = nn.LayerNorm(dim), nn.Dropout(head_drop), nn.Linear(dim, n_cls)

    def forward(self, x, valid):
        m = valid[:, :, None].to(x.dtype)
        h = self.stem_norm(self.stem(x)) * m
        for b in self.blocks:
            h = b(h, valid)
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        return self.head(self.drop(self.norm(pooled)))


class Exportable(nn.Module):
    """One un-padded clip in, logits out — what the browser runs."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x):                        # 1×T×N_FEAT
        return self.net(x, torch.ones(x.shape[:2], dtype=torch.bool, device=x.device))


# ---------------------------------------------------------------- metrics

def scores(logits: np.ndarray, y: np.ndarray) -> dict:
    """top-1/5, and macro-F1 / balanced accuracy over the classes present in y."""
    pred = logits.argmax(1)
    top5 = np.argsort(-logits, 1)[:, :5]
    f1s, recalls = [], []
    for c in np.unique(y):
        tp = int(((pred == c) & (y == c)).sum())
        fp = int(((pred == c) & (y != c)).sum())
        fn = int(((pred != c) & (y == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * p * r / (p + r) if p + r else 0.0)
        recalls.append(r)
    return {'top1': float((pred == y).mean()), 'top5': float((top5 == y[:, None]).any(1).mean()),
            'macro_f1': float(np.mean(f1s)), 'balanced_acc': float(np.mean(recalls)), 'n': int(len(y))}


@torch.no_grad()
def predict(model, bank, device, batch=128, workers=2, tta=False, groups=None, face=True):
    model.eval()
    outs = []
    for flip in ([False, True] if tta else [False]):
        dl = torch.utils.data.DataLoader(SignDataset(bank, flip=flip, groups=groups, face=face), batch_size=batch, shuffle=False,
                                         num_workers=workers, collate_fn=collate)
        logits = []
        for x, m, _ in dl:
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                logits.append(model(x.to(device), m.to(device)).float().softmax(1).cpu())
        outs.append(torch.cat(logits).numpy())
    return np.mean(outs, 0)


def reads_face(groups) -> bool:
    """Does the model input change when the face is missing? Also true for 'flags' alone: one of the
    four presence flags is "face detected", so a hands + pose model still sees the face tracker."""
    return not groups or bool({'face', 'blend', 'flags'} & set(groups))


# ---------------------------------------------------------------- training

def fit(cfg, signs, words, vocab_list, out_dir, log=print):
    """signs/words: dict split → Bank. Writes history.csv, best.pt, metrics.json, … to out_dir."""
    cfg = {**DEFAULTS, **cfg}
    os.makedirs(out_dir, exist_ok=True)
    torch.manual_seed(cfg['seed'])
    np.random.seed(cfg['seed'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    C = len(vocab_list)
    tr = signs['train']
    counts = np.bincount(tr.labels, minlength=C)
    weights = counts[tr.labels].astype(np.float64) ** -cfg['balance_power']
    sampler = torch.utils.data.WeightedRandomSampler(torch.as_tensor(weights), num_samples=len(tr), replacement=True)
    cols, groups = FT.columns(cfg['groups']), cfg['groups']
    dl = torch.utils.data.DataLoader(SignDataset(tr, train=True, groups=groups, face_drop=cfg['face_drop']),
                                     batch_size=cfg['batch'], sampler=sampler,
                                     num_workers=cfg['workers'], collate_fn=collate, drop_last=True,
                                     persistent_workers=cfg['workers'] > 0, pin_memory=device.type == 'cuda')
    evals = {k: b for k, b in (('val_words', words.get('val')), ('val_signs', signs.get('val'))) if b is not None and len(b)}
    major = int(np.bincount(tr.labels, minlength=C).argmax())

    model = Net(len(cols), C, cfg['dim'], cfg['stages'], cfg['convs'], cfg['kernel'], cfg['heads'],
                cfg['drop'], cfg['head_drop']).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    steps, warm = cfg['epochs'] * len(dl), max(1, cfg['warmup'] * len(dl))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (
        1 + math.cos(math.pi * min(1.0, max(0, s - warm) / max(1, steps - warm)))))
    scaler = torch.amp.GradScaler(enabled=device.type == 'cuda')

    log(f'{device} | {n_params / 1e6:.2f} M params | {len(cols)} features | {len(tr)} train signs | {C} classes | '
        f'{len(dl)} steps/epoch | majority class "{vocab_list[major]}"')
    for k, b in evals.items():
        log(f'  {k}: {len(b)} samples, majority-class accuracy {(b.labels == major).mean():.3f}')
    hist_path = os.path.join(out_dir, 'history.csv')
    history, best, best_ep = [], -1.0, 0
    for ep in range(1, cfg['epochs'] + 1):
        model.train()
        t0, tl, tc, tn = time.time(), 0.0, 0, 0
        for x, m, y in dl:
            x, m, y = x.to(device, non_blocking=True), m.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                logits = model(x, m)
                loss = F.cross_entropy(logits.float(), y, label_smoothing=cfg['label_smoothing'])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            tl, tc, tn = tl + loss.item() * len(y), tc + int((logits.argmax(1) == y).sum()), tn + len(y)
        row = {'epoch': ep, 'lr': opt.param_groups[0]['lr'], 'train_loss': tl / tn, 'train_acc': tc / tn}
        for k, b in evals.items():
            s = scores(predict(model, b, device, workers=min(2, cfg['workers']), groups=groups), b.labels)
            row.update({f'{k}_{m}': v for m, v in s.items() if m != 'n'})
        row['sec'] = time.time() - t0
        key = 'val_words_macro_f1' if 'val_words' in evals else 'val_signs_macro_f1'
        row['best'] = row.get(key, 0) > best
        if row['best']:
            best, best_ep = row[key], ep
            torch.save({'model': model.state_dict(), 'cfg': cfg, 'vocab': vocab_list, 'epoch': ep,
                        'feature_version': FT.FEATURE_VERSION}, os.path.join(out_dir, 'best.pt'))
        history.append(row)
        with open(hist_path, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(row))
            w.writeheader()
            w.writerows(history)
        log(_line(row, ep == 1))

    # test once, with the best-on-val weights (plain and with mirror TTA)
    ck = torch.load(os.path.join(out_dir, 'best.pt'), map_location=device)
    model.load_state_dict(ck['model'])
    result = {'best_epoch': best_ep, 'params': n_params, 'n_features': int(len(cols)), 'cfg': cfg,
              'majority_class': vocab_list[major]}
    preds = {}
    for name, bank in (('test_words', words.get('test')), ('test_signs', signs.get('test')),
                       ('val_words', words.get('val')), ('val_signs', signs.get('val'))):
        if bank is None or not len(bank):
            continue
        nw = min(2, cfg['workers'])
        p = predict(model, bank, device, workers=nw, groups=groups)
        preds.update({f'{name}_probs': p.astype(np.float16), f'{name}_labels': bank.labels,
                      f'{name}_ids': np.array(bank.ids)})
        result[name] = scores(p, bank.labels)
        result[name]['majority_acc'] = float((bank.labels == major).mean())
        result[name + '_tta'] = scores(predict(model, bank, device, workers=nw, tta=True, groups=groups), bank.labels)
        if reads_face(groups):                                      # privacy mode: same model, no face points
            result[name + '_noface'] = scores(predict(model, bank, device, workers=nw, groups=groups, face=False),
                                              bank.labels)
        if name == 'test_words':
            _per_class(p, bank.labels, vocab_list, out_dir)
    with open(os.path.join(out_dir, 'metrics.json'), 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=1)
    with open(os.path.join(out_dir, 'vocab.json'), 'w', encoding='utf-8') as f:
        json.dump(vocab_list, f, ensure_ascii=False)
    # per-sample probabilities, for analyses without retraining (e.g. merging inflected word forms)
    np.savez_compressed(os.path.join(out_dir, 'predictions.npz'), **preds)
    ok = export_onnx(model, os.path.join(out_dir, 'model.onnx'), log, n_in=len(cols))
    write_manifest(out_dir, cfg, vocab_list, result, n_in=len(cols), onnx_ok=ok)
    return result


def write_manifest(out_dir, cfg, vocab_list, result, n_in, onnx_ok=True):
    """manifest.json next to model.onnx: everything the browser or the landmark server needs to feed
    the model and to refuse a mismatch (feature_version), plus where the model came from."""
    m = {
        'feature_version': FT.FEATURE_VERSION,
        'model_file': 'model.onnx' if onnx_ok else None, 'opset': 17,
        'labels': vocab_list,
        'input': {'name': 'x', 'shape': [1, 'T', n_in], 'dtype': 'float32', 'n_features': n_in,
                  'groups': cfg['groups'] or list(FT.GROUPS), 'max_len': FT.MAX_LEN,
                  'padding': 'none — one clip per call (T frames, T <= max_len after preprocess)'},
        'output': {'name': 'logits', 'shape': [1, len(vocab_list)], 'note': 'softmax not applied; '
                   'ensembles average the softmax probabilities'},
        # without face = trained with the face randomly removed, or an input the face cannot change
        'face_drop': cfg['face_drop'], 'works_without_face': bool(cfg['face_drop']) or not reads_face(cfg['groups']),
        'contract': FT.spec(),
        'run': os.path.basename(os.path.normpath(out_dir)), 'seed': cfg['seed'], 'fold': cfg.get('fold'),
        'split_id': cfg.get('split_id'), 'best_epoch': result.get('best_epoch'), 'params': result.get('params'),
        'metrics': {k: v for k, v in result.items() if k.startswith(('test_', 'val_'))},
        'cfg': cfg,
    }
    with open(os.path.join(out_dir, 'manifest.json'), 'w', encoding='utf-8') as f:
        json.dump(m, f, ensure_ascii=False, indent=1)


def fit_many(runs: dict, signs, words, vocab_list, out_root, log=print):
    """Several configurations on the same loaded data, one after another → out_root/<name>/ +
    summary.csv. A failing configuration is logged and skipped, the others still run."""
    rows = []
    for name, cfg in runs.items():
        log(f'\n===== {name}: {cfg}')
        t0 = time.time()
        try:
            r = fit(cfg, signs, words, vocab_list, os.path.join(out_root, name), log)
        except Exception as e:  # noqa: BLE001
            log(f'{name} FAILED: {e!r}')
            rows.append({'run': name, 'error': repr(e)[:200]})
            continue
        row = {'run': name, 'cfg': json.dumps(cfg), 'best_epoch': r['best_epoch'], 'params': r['params'],
               'n_features': r['n_features'], 'minutes': round((time.time() - t0) / 60, 1)}
        for k in ('test_words', 'test_signs', 'val_words', 'test_words_noface', 'test_signs_noface'):
            if k in r:
                row.update({f'{k}_{m}': round(r[k][m], 4) for m in ('top1', 'top5', 'macro_f1', 'balanced_acc')})
        row['test_words_majority'] = round(r['test_words']['majority_acc'], 4) if 'test_words' in r else None
        rows.append(row)
        with open(os.path.join(out_root, 'summary.csv'), 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=sorted({k for rr in rows for k in rr}, key=lambda k: (k != 'run', k)))
            w.writeheader()
            w.writerows(rows)
    return rows


def _line(row, header):
    cols = [('epoch', 'ep', '{:>3d}'), ('train_loss', 'loss', '{:6.3f}'), ('train_acc', 'tr_acc', '{:6.3f}'),
            ('val_words_top1', 'W_top1', '{:6.3f}'), ('val_words_top5', 'W_top5', '{:6.3f}'),
            ('val_words_macro_f1', 'W_mF1', '{:6.3f}'), ('val_signs_top1', 'S_top1', '{:6.3f}'),
            ('val_signs_macro_f1', 'S_mF1', '{:6.3f}'), ('sec', 'sec', '{:5.0f}')]
    cols = [c for c in cols if c[0] in row]
    s = '  '.join(f.format(row[k]) for k, _, f in cols) + ('  *' if row.get('best') else '')
    return ('  '.join(f'{h:>{len(f.format(row[k]))}}' for k, h, f in cols) + '\n' + s) if header else s


def _per_class(p, y, vocab, out_dir):
    pred = p.argmax(1)
    rows = []
    for c in np.unique(y):
        tp = int(((pred == c) & (y == c)).sum())
        n, npred = int((y == c).sum()), int((pred == c).sum())
        confused = np.bincount(pred[(y == c) & (pred != c)], minlength=len(vocab))
        rows.append({'gloss': vocab[c], 'n': n, 'recall': tp / n, 'precision': tp / npred if npred else 0.0,
                     'most_confused_with': vocab[int(confused.argmax())] if confused.any() else ''})
    with open(os.path.join(out_dir, 'per_class_test_words.csv'), 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(sorted(rows, key=lambda r: r['recall']))


def export_onnx(model, path, log=print, n_in=FT.N_FEAT):
    try:
        model = model.float().eval().cpu()
        dummy = torch.zeros(1, 32, n_in)
        kw = dict(input_names=['x'], output_names=['logits'], dynamic_axes={'x': {1: 'T'}}, opset_version=17)
        try:
            torch.onnx.export(Exportable(model), (dummy,), path, dynamo=False, **kw)
        except TypeError:
            torch.onnx.export(Exportable(model), (dummy,), path, **kw)
        log(f'ONNX: {path} ({os.path.getsize(path) / 1e6:.1f} MB)')
        ok = True
        try:                                     # same logits as torch, for a short and a long clip?
            import onnxruntime as ort
            sess = ort.InferenceSession(path)
            for T_ in (12, 64):
                x = torch.randn(1, T_, n_in)
                ref = Exportable(model)(x).detach().numpy()
                got = sess.run(None, {'x': x.numpy()})[0]
                log(f'  onnxruntime vs torch, T={T_}: max abs diff {float(np.abs(ref - got).max()):.2e}')
        except ImportError:
            log('  (onnxruntime not installed — not compared)')
        return ok
    except Exception as e:  # noqa: BLE001 — the browser export must not lose a finished training run
        log(f'ONNX export failed (training results are saved): {e!r}')
        return False

In [ ]:
import pandas as pd, torch
from azsl import train as T
INPUT = '/kaggle/input' if ON_KAGGLE else OUT

def find(pattern):
    return sorted({os.path.dirname(p) for p in glob.glob(os.path.join(INPUT, '**', pattern), recursive=True)})

splits_dir = find('splits/words200_split.csv')[0]
words_dirs = find('landmarks/words200/manifest.csv')
sentence_dirs = find('landmarks/sentences_cam2/shard*/manifest.csv')
print('splits:', splits_dir, '\nwords:', words_dirs, '\nsentence shards:', len(sentence_dirs))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — this will be slow')
with open(os.path.join(splits_dir, 'benchmark.json'), encoding='utf-8') as f:
    print('split:', json.load(f)['signers'].get('split_groups'))

In [ ]:
wsplit = pd.read_csv(os.path.join(splits_dir, 'words200_split.csv'), dtype={'key': str, 'rec_id': str})
tags = pd.read_csv(os.path.join(splits_dir, 'glosses_split.csv'), dtype={'key': str, 'rec_id': str})
import hashlib
with open(os.path.join(splits_dir, 'words200_split.csv'), 'rb') as f:
    CFG['split_id'] = hashlib.sha1(f.read()).hexdigest()[:10]     # recorded in every model manifest
vocab_list = sorted(wsplit.gloss.unique())
vocab = {g: i for i, g in enumerate(vocab_list)}
tags = tags[tags.in_w200 & tags.has_landmarks & tags.split.notna()]
if SMOKE:
    tags = tags.groupby('split', group_keys=False).apply(lambda d: d.sample(min(len(d), 1500), random_state=0))
    wsplit = wsplit.groupby('split', group_keys=False).apply(lambda d: d.sample(min(len(d), 300), random_state=0))
    CFG.update(epochs=2)
    RUNS = dict(list(RUNS.items())[:2])
signs = T.load_sentence_signs(sentence_dirs, tags, vocab, margin=CFG['margin'])
words = T.load_words200(words_dirs, wsplit, vocab)

### Training — one line per epoch per run (`*` = new best on val macro-F1)

In [ ]:
summary = T.fit_many({name: {**CFG, **change} for name, change in RUNS.items()},
                     signs, words, vocab_list, os.path.join(OUT, 'train'))

### Test (scored once per run, best-on-val checkpoint)

In [ ]:
import matplotlib.pyplot as plt
summ = pd.read_csv(os.path.join(OUT, 'train', 'summary.csv'))
cols = [c for c in ('run', 'best_epoch', 'n_features', 'params', 'minutes', 'val_words_top1', 'test_words_top1',
                    'test_words_top5', 'test_words_macro_f1', 'test_words_balanced_acc', 'test_signs_top1',
                    'test_signs_macro_f1', 'test_words_majority', 'error') if c in summ]
display(summ[cols])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name in RUNS:
    path = os.path.join(OUT, 'train', name, 'history.csv')
    if os.path.exists(path):
        h = pd.read_csv(path)
        ax[0].plot(h.epoch, h.val_words_top1, label=name)
        ax[1].plot(h.epoch, h.val_words_macro_f1, label=name)
ax[0].set_title('val Words_200 top-1'); ax[1].set_title('val Words_200 macro-F1')
ax[0].legend(); plt.show()